# source code notebook

This notebook is a codebook for the project. The scripts in `src/` are still the files I would edit first, but this notebook keeps the same code in notebook cells for reading, handoff, or classroom-style walkthroughs.

The cells use `%%writefile`, so running one of them writes that script back to `../src/`. If you only want to run the analysis, use `copenhagen_osm_accessibility_pipeline.ipynb` instead.

In [1]:
from pathlib import Path
import os

if Path.cwd().name != "notebooks":
    notebooks_dir = Path.cwd() / "notebooks"
    if notebooks_dir.exists():
        os.chdir(notebooks_dir)

print(f"Notebook working directory: {Path.cwd()}")


Notebook working directory: c:\Users\dubst\Desktop\DataScience\Geospatial\Project\notebooks


## `config.py`

Project paths, CRS choices, thresholds, and the file names used across the pipeline. I keep these in one place so the numbered scripts do not quietly drift apart.

In [2]:
%%writefile ../src/config.py
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_OFFICIAL_DIR = RAW_DIR / "official"
RAW_OSM_DIR = RAW_DIR / "osm"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
MAPS_DIR = OUTPUTS_DIR / "maps"
REPORT_DIR = PROJECT_ROOT / "report"

PBF_PATH = RAW_OSM_DIR / "denmark-latest.osm.pbf"

CRS_WGS84 = "EPSG:4326"
CRS_METRIC = "EPSG:25832"
CRS_WEB_MERCATOR = "EPSG:3857"

GRID_SIZE_M = 500
WALKING_SPEED_KMH = 5.0
ACCESS_THRESHOLD_MINUTES = 15.0
SNAP_WARNING_DISTANCE_M = 100.0

CKAN_API_BASE = "https://admin.opendata.dk/api/3/action"
CKAN_ORGANIZATION = "city-of-copenhagen"

OFFICIAL_DATASETS = {
    "libraries": {
        "dataset_name": "Biblioteker",
        "package_ids": ["biblioteker"],
        "search_query": "title:Biblioteker organization:city-of-copenhagen",
        "amenity_type": "library",
    },
    "playgrounds": {
        "dataset_name": "Legepladser",
        "package_ids": ["legepladser1"],
        "search_query": "title:Legepladser organization:city-of-copenhagen tags:legepladser",
        "amenity_type": "playground",
    },
    "sports_facilities": {
        "dataset_name": "Idraetsanlaeg",
        "package_ids": ["idraetsanlaeg"],
        "search_query": "title:Idraetsanlaeg organization:city-of-copenhagen",
        "amenity_type": "sports_facility",
    },
    "bydele": {
        "dataset_name": "Bydele",
        "package_ids": ["bydele"],
        "search_query": "title:Bydele organization:city-of-copenhagen",
        "amenity_type": None,
    },
}

AMENITY_LAYER_KEYS = ["libraries", "playgrounds", "sports_facilities"]

AMENITY_TYPE_BY_LAYER = {
    "libraries": "library",
    "playgrounds": "playground",
    "sports_facilities": "sports_facility",
}

LAYER_BY_AMENITY_TYPE = {v: k for k, v in AMENITY_TYPE_BY_LAYER.items()}

MATCH_THRESHOLDS_M = {
    "library": 100.0,
    "playground": 100.0,
    "sports_facility": 150.0,
}

OFFICIAL_CLEAN_FILES = {
    "libraries": PROCESSED_DIR / "official_libraries.gpkg",
    "playgrounds": PROCESSED_DIR / "official_playgrounds.gpkg",
    "sports_facilities": PROCESSED_DIR / "official_sports_facilities.gpkg",
}

OSM_RAW_FILES = {
    "libraries": PROCESSED_DIR / "osm_libraries.gpkg",
    "playgrounds": PROCESSED_DIR / "osm_playgrounds.gpkg",
    "sports_facilities": PROCESSED_DIR / "osm_sports_facilities.gpkg",
}

OSM_CLEAN_FILES = {
    "libraries": PROCESSED_DIR / "osm_libraries_clean.gpkg",
    "playgrounds": PROCESSED_DIR / "osm_playgrounds_clean.gpkg",
    "sports_facilities": PROCESSED_DIR / "osm_sports_facilities_clean.gpkg",
}

BYDELE_FILE = PROCESSED_DIR / "bydele.gpkg"
BOUNDARY_FILE = PROCESSED_DIR / "copenhagen_boundary.gpkg"

OSM_WALKING_EDGES_FILE = PROCESSED_DIR / "osm_walking_edges.gpkg"
OSM_WALKING_NODES_FILE = PROCESSED_DIR / "osm_walking_nodes.gpkg"
OSM_WALKING_EDGES_CLEAN_FILE = PROCESSED_DIR / "osm_walking_edges_clean.gpkg"
OSM_WALKING_NODES_CLEAN_FILE = PROCESSED_DIR / "osm_walking_nodes_clean.gpkg"
OSM_WALKING_GRAPH_FILE = PROCESSED_DIR / "osm_walking_graph.graphml"

ORIGINS_GRID_FILE = PROCESSED_DIR / "origins_grid_500m.gpkg"
ORIGINS_POINTS_FILE = PROCESSED_DIR / "origins_points_500m.gpkg"
ORIGINS_POINTS_SNAPPED_FILE = PROCESSED_DIR / "origins_points_500m_snapped.gpkg"

ACCESSIBILITY_TABLE = TABLES_DIR / "origin_accessibility_comparison.csv"
ACCESSIBILITY_GPKG = PROCESSED_DIR / "origin_accessibility_comparison.gpkg"
CLASSIFIED_ACCESSIBILITY_TABLE = TABLES_DIR / "origin_accessibility_classified.csv"
CLASSIFIED_ACCESSIBILITY_GPKG = PROCESSED_DIR / "origin_accessibility_classified.gpkg"
COMPOSITE_DISAGREEMENT_TABLE = TABLES_DIR / "composite_disagreement_scores.csv"
COMPOSITE_DISAGREEMENT_GPKG = PROCESSED_DIR / "composite_disagreement.gpkg"


def ensure_directories() -> None:
    for path in [
        RAW_OFFICIAL_DIR,
        RAW_OSM_DIR,
        PROCESSED_DIR,
        FIGURES_DIR,
        TABLES_DIR,
        MAPS_DIR,
        REPORT_DIR,
    ]:
        path.mkdir(parents=True, exist_ok=True)


Overwriting ../src/config.py


## `utils.py`

Shared helpers for reading vector files, cleaning geometries, snapping points, and writing outputs. It is not glamorous code, but it keeps the rest of the project shorter.

In [3]:
%%writefile ../src/utils.py
from __future__ import annotations

import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import pandas as pd

from config import CRS_METRIC, CRS_WGS84, PROCESSED_DIR


NAME_CANDIDATES = [
    "name",
    "navn",
    "titel",
    "title",
    "lokalitet",
    "facilitet",
    "bibliotek",
    "legeplads",
    "halnavn",
]

ID_CANDIDATES = [
    "amenity_id",
    "original_id",
    "id",
    "fid",
    "objectid",
    "ogc_fid",
    "osm_id",
    "element_id",
    "uuid",
    "globalid",
]

CATEGORY_CANDIDATES = [
    "amenity",
    "leisure",
    "type",
    "kategori",
    "category",
    "klasse",
    "funktion",
    "facilitetstype",
]

DISTRICT_NAME_CANDIDATES = [
    "bydel",
    "navn",
    "name",
    "district",
    "district_name",
    "bydelsnavn",
]


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def slugify(value: object, fallback: str = "value") -> str:
    text = str(value or fallback).strip().lower()
    text = text.replace("ae", "ae").replace("oe", "oe").replace("aa", "aa")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or fallback


def require_file(path: Path, message: str | None = None) -> Path:
    if not path.exists():
        raise FileNotFoundError(message or f"Required file not found: {path}")
    return path


def import_geopandas():
    try:
        import geopandas as gpd
    except ImportError as exc:
        raise ImportError(
            "This script requires geopandas. Install dependencies with "
            "`python -m pip install -r requirements.txt` or the conda commands in README.md."
        ) from exc
    return gpd


def import_shapely_geometry():
    try:
        from shapely import wkt
        from shapely.geometry import Point, box
        from shapely.ops import unary_union
    except ImportError as exc:
        raise ImportError(
            "This script requires shapely. Install dependencies from requirements.txt."
        ) from exc
    return Point, box, unary_union, wkt


def infer_crs_from_bounds(gdf) -> str:
    if gdf.empty:
        return CRS_WGS84
    minx, miny, maxx, maxy = gdf.total_bounds
    if all(math.isfinite(v) for v in [minx, miny, maxx, maxy]):
        if -180 <= minx <= 180 and -90 <= miny <= 90 and -180 <= maxx <= 180 and -90 <= maxy <= 90:
            return CRS_WGS84
    return CRS_METRIC


def ensure_crs(gdf, default: str | None = None):
    if gdf.crs is None:
        gdf = gdf.set_crs(default or infer_crs_from_bounds(gdf), allow_override=True)
    return gdf


def make_valid_geometries(gdf):
    if gdf.empty:
        return gdf
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna()].copy()
    try:
        gdf["geometry"] = gdf.geometry.make_valid()
    except Exception:
        gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf


def find_column(columns: Iterable[str], candidates: Iterable[str]) -> str | None:
    lower_map = {str(col).lower(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    for candidate in candidates:
        key = candidate.lower()
        for lower, original in lower_map.items():
            if key in lower:
                return original
    return None


def read_metadata_table(path: Path) -> pd.DataFrame:
    require_file(path)
    return pd.read_csv(path)


def get_downloaded_resource(metadata: pd.DataFrame, dataset_key: str) -> Path:
    row = metadata.loc[metadata["dataset_key"] == dataset_key]
    if row.empty:
        raise ValueError(f"No downloaded official resource is recorded for {dataset_key}.")
    path = Path(row.iloc[0]["downloaded_file"])
    if not path.is_absolute():
        path = Path.cwd() / path
    return require_file(path)


def read_vector_any(path: Path, data_format: str | None = None):
    gpd = import_geopandas()
    Point, _, _, wkt = import_shapely_geometry()
    path = Path(path)
    suffix = path.suffix.lower().lstrip(".")
    fmt = (data_format or suffix).lower()

    if suffix == "csv" or fmt == "csv":
        df = pd.read_csv(path)
        geometry_col = find_column(df.columns, ["geometry", "geom", "wkt", "the_geom"])
        if geometry_col:
            geometries = df[geometry_col].apply(lambda val: wkt.loads(val) if pd.notna(val) else None)
            gdf = gpd.GeoDataFrame(df, geometry=geometries, crs=CRS_WGS84)
            return ensure_crs(gdf)

        lon_col = find_column(
            df.columns,
            ["lon", "lng", "longitude", "x", "xcoord", "x_koord", "koord_x", "easting"],
        )
        lat_col = find_column(
            df.columns,
            ["lat", "latitude", "y", "ycoord", "y_koord", "koord_y", "northing"],
        )
        if not lon_col or not lat_col:
            raise ValueError(
                f"CSV {path} has no geometry/WKT column and no recognizable coordinate columns. "
                f"Columns: {list(df.columns)}"
            )

        coords = df[[lon_col, lat_col]].apply(pd.to_numeric, errors="coerce")
        geometries = [Point(xy) if pd.notna(xy[0]) and pd.notna(xy[1]) else None for xy in coords.to_numpy()]
        gdf = gpd.GeoDataFrame(df, geometry=geometries)
        return ensure_crs(gdf)

    try:
        gdf = gpd.read_file(path)
    except Exception:
        if suffix == "zip":
            gdf = gpd.read_file(f"zip://{path}")
        else:
            raise
    return ensure_crs(gdf)


def safe_write_gdf(gdf, path: Path, layer: str | None = None) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    kwargs = {"driver": "GPKG"}
    if layer:
        kwargs["layer"] = layer
    gdf.to_file(path, **kwargs)


def safe_write_csv(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_json(data: object, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def dissolve_to_boundary(gdf):
    gpd = import_geopandas()
    gdf = make_valid_geometries(ensure_crs(gdf)).to_crs(CRS_METRIC)
    if hasattr(gdf.geometry, "union_all"):
        geom = gdf.geometry.union_all()
    else:
        geom = gdf.geometry.unary_union
    return gpd.GeoDataFrame({"name": ["Copenhagen Municipality"]}, geometry=[geom], crs=CRS_METRIC)


def load_boundary(metric: bool = True):
    gpd = import_geopandas()
    boundary_path = PROCESSED_DIR / "copenhagen_boundary.gpkg"
    bydele_path = PROCESSED_DIR / "bydele.gpkg"
    if boundary_path.exists():
        boundary = gpd.read_file(boundary_path)
    elif bydele_path.exists():
        boundary = dissolve_to_boundary(gpd.read_file(bydele_path))
    else:
        raise FileNotFoundError(
            "Copenhagen boundary not found. Run 01_download_official_data.py and "
            "03_clean_official_amenities.py first."
        )
    boundary = ensure_crs(boundary)
    return boundary.to_crs(CRS_METRIC if metric else CRS_WGS84)


def clip_to_boundary(gdf, boundary):
    gpd = import_geopandas()
    if gdf.empty:
        return gdf
    gdf = make_valid_geometries(ensure_crs(gdf)).to_crs(boundary.crs)
    boundary = make_valid_geometries(ensure_crs(boundary)).to_crs(gdf.crs)
    try:
        clipped = gpd.clip(gdf, boundary)
    except Exception:
        clipped = gdf[gdf.intersects(boundary.geometry.iloc[0])].copy()
    return clipped.reset_index(drop=True)


def representative_points(gdf):
    gdf = gdf.copy()
    geometries = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            geometries.append(None)
        elif geom.geom_type in {"Point", "MultiPoint"}:
            geometries.append(geom.representative_point())
        elif geom.geom_type in {"LineString", "MultiLineString"}:
            geometries.append(geom.interpolate(0.5, normalized=True))
        else:
            geometries.append(geom.representative_point())
    gdf["geometry"] = geometries
    return make_valid_geometries(gdf)


def standardize_amenities(gdf, amenity_type: str, source: str, dataset_label: str, id_prefix: str):
    gpd = import_geopandas()
    gdf = make_valid_geometries(ensure_crs(gdf))
    if gdf.empty:
        return gpd.GeoDataFrame(
            columns=[
                "amenity_id",
                "amenity_type",
                "source",
                "name",
                "original_category",
                "original_id",
                "longitude",
                "latitude",
                "geometry",
            ],
            geometry="geometry",
            crs=CRS_METRIC,
        )

    name_col = find_column(gdf.columns, NAME_CANDIDATES)
    id_col = find_column(gdf.columns, ID_CANDIDATES)
    category_col = find_column(gdf.columns, CATEGORY_CANDIDATES)

    points = representative_points(gdf).to_crs(CRS_METRIC).reset_index(drop=True)
    lonlat = points.to_crs(CRS_WGS84)

    names = points[name_col].astype(str) if name_col else pd.Series([""] * len(points))
    original_ids = points[id_col].astype(str) if id_col else pd.Series(points.index.astype(str))
    categories = points[category_col].astype(str) if category_col else pd.Series([dataset_label] * len(points))

    out = gpd.GeoDataFrame(
        {
            "amenity_id": [f"{id_prefix}_{i + 1:05d}" for i in range(len(points))],
            "amenity_type": amenity_type,
            "source": source,
            "name": names.fillna("").replace("nan", ""),
            "original_category": categories.fillna(dataset_label).replace("nan", dataset_label),
            "original_id": original_ids.fillna("").replace("nan", ""),
            "longitude": lonlat.geometry.x,
            "latitude": lonlat.geometry.y,
        },
        geometry=points.geometry,
        crs=CRS_METRIC,
    )

    out["_name_norm"] = out["name"].fillna("").str.lower().str.strip()
    out["_x_round"] = out.geometry.x.round(1)
    out["_y_round"] = out.geometry.y.round(1)
    out = out.drop_duplicates(subset=["source", "amenity_type", "_name_norm", "_x_round", "_y_round"])
    out = out.drop(columns=["_name_norm", "_x_round", "_y_round"]).reset_index(drop=True)
    out["amenity_id"] = [f"{id_prefix}_{i + 1:05d}" for i in range(len(out))]
    return out[
        [
            "amenity_id",
            "amenity_type",
            "source",
            "name",
            "original_category",
            "original_id",
            "longitude",
            "latitude",
            "geometry",
        ]
    ]


def inspect_gdf(dataset_name: str, gdf) -> dict:
    geom_types = []
    if "geometry" in gdf:
        geom_types = sorted([str(value) for value in gdf.geometry.geom_type.dropna().unique()])
    return {
        "dataset_name": dataset_name,
        "crs": str(gdf.crs),
        "geometry_type": ";".join(geom_types),
        "row_count": int(len(gdf)),
        "column_names": "|".join([str(col) for col in gdf.columns]),
        "missing_geometry_count": int(gdf.geometry.isna().sum()) if "geometry" in gdf else len(gdf),
    }


def resolve_node_id_column(nodes) -> str:
    for col in ["node_id", "id", "osm_id", "osmid"]:
        if col in nodes.columns:
            return col
    raise ValueError(f"Could not identify node ID column. Columns: {list(nodes.columns)}")


def resolve_edge_endpoint_columns(edges) -> tuple[str, str]:
    candidates = [("u", "v"), ("from", "to"), ("source", "target")]
    for left, right in candidates:
        if left in edges.columns and right in edges.columns:
            return left, right
    raise ValueError(f"Could not identify edge endpoint columns. Columns: {list(edges.columns)}")


def nearest_node_snap(points, nodes, point_id_col: str = "amenity_id") -> pd.DataFrame:
    try:
        from scipy.spatial import cKDTree
    except ImportError as exc:
        raise ImportError("Snapping requires scipy. Install dependencies from requirements.txt.") from exc

    points = ensure_crs(points).to_crs(CRS_METRIC)
    nodes = ensure_crs(nodes).to_crs(CRS_METRIC)
    node_col = resolve_node_id_column(nodes)

    node_coords = list(zip(nodes.geometry.x, nodes.geometry.y))
    tree = cKDTree(node_coords)
    point_coords = list(zip(points.geometry.x, points.geometry.y))
    distances, indices = tree.query(point_coords, k=1)

    ids = points[point_id_col].astype(str).to_numpy() if point_id_col in points.columns else points.index.astype(str).to_numpy()
    snapped = pd.DataFrame(
        {
            point_id_col: ids,
            "nearest_node": nodes.iloc[indices][node_col].astype(str).to_numpy(),
            "snap_distance_m": distances,
        }
    )
    return snapped


def finite_or_none(value: object) -> float | None:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def clean_numeric_difference(osm_time: object, official_time: object) -> float | None:
    osm_value = finite_or_none(osm_time)
    official_value = finite_or_none(official_time)
    if osm_value is None or official_value is None:
        return None
    return osm_value - official_value


def classify_accessibility(official_access: bool, osm_access: bool) -> str:
    if official_access and osm_access:
        return "agreement_accessible"
    if (not official_access) and (not osm_access):
        return "agreement_inaccessible"
    if (not official_access) and osm_access:
        return "osm_false_access"
    return "osm_hidden_access"


Overwriting ../src/utils.py


## `origin_quality.py`

Helpers for the grid-quality fix: largest-overlap district assignment and origin snapping flags.

In [4]:
%%writefile ../src/origin_quality.py
from __future__ import annotations

import pandas as pd

from config import (
    BYDELE_FILE,
    CRS_METRIC,
    ORIGINS_POINTS_SNAPPED_FILE,
)
from utils import (
    DISTRICT_NAME_CANDIDATES,
    find_column,
    import_geopandas,
    nearest_node_snap,
    require_file,
    resolve_node_id_column,
)


MIN_DISTRICT_OVERLAP_SHARE = 0.10


def load_districts():
    gpd = import_geopandas()
    bydele = gpd.read_file(require_file(BYDELE_FILE)).to_crs(CRS_METRIC)
    district_col = find_column(bydele.columns, DISTRICT_NAME_CANDIDATES)
    if district_col is None:
        bydele = bydele.reset_index().rename(columns={"index": "district"})
        district_col = "district"
    districts = bydele[[district_col, "geometry"]].rename(columns={district_col: "district"}).copy()
    districts["district"] = districts["district"].astype(str)
    return districts


def dissolved_boundary(districts):
    gpd = import_geopandas()
    if hasattr(districts.geometry, "union_all"):
        geometry = districts.geometry.union_all()
    else:
        geometry = districts.geometry.unary_union
    return gpd.GeoDataFrame({"name": ["Copenhagen Municipality"]}, geometry=[geometry], crs=districts.crs)


def previous_centroid_assignment(points, districts) -> pd.DataFrame:
    """Reconstruct the earlier centroid-based district assignment for diagnostics."""
    gpd = import_geopandas()
    points = points.to_crs(CRS_METRIC)
    districts = districts.to_crs(CRS_METRIC)
    joined = gpd.sjoin(
        points[["origin_id", "geometry"]],
        districts[["district", "geometry"]],
        how="left",
        predicate="intersects",
    )[["origin_id", "district"]]
    joined = joined.drop_duplicates(subset=["origin_id"]).rename(
        columns={"district": "previous_district_assignment_if_available"}
    )
    boundary = dissolved_boundary(districts)
    within = points[["origin_id", "geometry"]].copy()
    within["centroid_within_any_district"] = within.geometry.within(boundary.geometry.iloc[0])
    return joined.merge(within[["origin_id", "centroid_within_any_district"]], on="origin_id", how="right")


def calculate_district_overlaps(grid, points, districts, min_overlap_share: float = MIN_DISTRICT_OVERLAP_SHARE):
    gpd = import_geopandas()
    grid = grid.to_crs(CRS_METRIC).copy()
    points = points.to_crs(CRS_METRIC).copy()
    districts = districts.to_crs(CRS_METRIC).copy()

    grid["origin_cell_area_m2"] = grid.geometry.area
    intersections = gpd.overlay(
        grid[["origin_id", "origin_cell_area_m2", "geometry"]],
        districts[["district", "geometry"]],
        how="intersection",
        keep_geom_type=True,
    )

    if intersections.empty:
        overlap_details = pd.DataFrame(
            columns=["origin_id", "district", "overlap_area_m2", "origin_cell_area_m2", "overlap_share"]
        )
    else:
        intersections["overlap_area_m2"] = intersections.geometry.area
        intersections["overlap_share"] = intersections["overlap_area_m2"] / intersections["origin_cell_area_m2"]
        overlap_details = pd.DataFrame(intersections.drop(columns="geometry"))

    ranked = overlap_details.sort_values(
        ["origin_id", "overlap_area_m2", "district"],
        ascending=[True, False, True],
    )
    top = ranked.drop_duplicates(subset=["origin_id"]).rename(
        columns={
            "district": "assigned_district",
            "overlap_area_m2": "largest_overlap_area_m2",
        }
    )

    assignments = grid[["origin_id", "origin_cell_area_m2", "geometry"]].merge(
        top[["origin_id", "assigned_district", "largest_overlap_area_m2", "overlap_share"]],
        on="origin_id",
        how="left",
    )
    assignments["largest_overlap_area_m2"] = assignments["largest_overlap_area_m2"].fillna(0.0)
    assignments["overlap_share"] = assignments["overlap_share"].fillna(0.0)
    assignments["low_overlap_edge_cell"] = assignments["overlap_share"] < min_overlap_share

    previous = previous_centroid_assignment(points, districts)
    assignments = assignments.merge(previous, on="origin_id", how="left")
    assignments["district_assignment_changed"] = (
        assignments["assigned_district"].fillna("")
        != assignments["previous_district_assignment_if_available"].fillna("")
    )

    boundary = dissolved_boundary(districts)
    municipality_overlap = gpd.overlay(
        grid[["origin_id", "origin_cell_area_m2", "geometry"]],
        boundary[["geometry"]],
        how="intersection",
        keep_geom_type=True,
    )
    if municipality_overlap.empty:
        municipality_shares = pd.DataFrame(columns=["origin_id", "municipality_overlap_share"])
    else:
        municipality_overlap["municipality_overlap_area_m2"] = municipality_overlap.geometry.area
        municipality_shares = (
            municipality_overlap.groupby("origin_id", as_index=False)["municipality_overlap_area_m2"]
            .sum()
            .merge(grid[["origin_id", "origin_cell_area_m2"]], on="origin_id", how="left")
        )
        municipality_shares["municipality_overlap_share"] = (
            municipality_shares["municipality_overlap_area_m2"] / municipality_shares["origin_cell_area_m2"]
        )
        municipality_shares = municipality_shares[["origin_id", "municipality_overlap_share"]]

    assignments = assignments.merge(municipality_shares, on="origin_id", how="left")
    assignments["municipality_overlap_share"] = assignments["municipality_overlap_share"].fillna(0.0)
    assignments["inferred_edge_or_harbour_cell"] = (
        (assignments["municipality_overlap_share"] < 0.95)
        | assignments["low_overlap_edge_cell"]
        | ~assignments["centroid_within_any_district"].fillna(False)
    )
    return assignments, overlap_details


def attach_assignment_to_points(points, assignments):
    columns = [
        "origin_id",
        "assigned_district",
        "largest_overlap_area_m2",
        "origin_cell_area_m2",
        "overlap_share",
        "low_overlap_edge_cell",
        "centroid_within_any_district",
        "previous_district_assignment_if_available",
        "district_assignment_changed",
        "municipality_overlap_share",
        "inferred_edge_or_harbour_cell",
    ]
    return points.drop(
        columns=[col for col in columns if col != "origin_id" and col in points.columns],
        errors="ignore",
    ).merge(assignments[columns], on="origin_id", how="left")


def load_existing_origin_snaps(points):
    if not ORIGINS_POINTS_SNAPPED_FILE.exists():
        return None
    gpd = import_geopandas()
    snapped = gpd.read_file(ORIGINS_POINTS_SNAPPED_FILE).to_crs(CRS_METRIC)
    columns = ["origin_id"]
    for col in ["nearest_node", "nearest_node_id", "snap_distance_m", "snap_flag_gt_100m"]:
        if col in snapped.columns:
            columns.append(col)
    snap = snapped[columns].copy()
    if "nearest_node" in snap.columns and "nearest_node_id" not in snap.columns:
        snap = snap.rename(columns={"nearest_node": "nearest_node_id"})
    return points.merge(snap, on="origin_id", how="left")


def calculate_origin_snaps(points, nodes):
    nodes = nodes.to_crs(CRS_METRIC).copy()
    node_col = resolve_node_id_column(nodes)
    nodes[node_col] = nodes[node_col].astype(str)
    snapped = nearest_node_snap(points.to_crs(CRS_METRIC), nodes, point_id_col="origin_id")
    snapped = snapped.rename(columns={"nearest_node": "nearest_node_id"})
    out = points.merge(snapped, on="origin_id", how="left")
    out["nearest_node_id"] = out["nearest_node_id"].astype(str)
    return out


def add_snap_flags(origins):
    out = origins.copy()
    out["snap_gt_50m"] = out["snap_distance_m"] > 50
    out["snap_gt_100m"] = out["snap_distance_m"] > 100
    out["snap_gt_250m"] = out["snap_distance_m"] > 250
    out["snap_gt_500m"] = out["snap_distance_m"] > 500
    return out


Overwriting ../src/origin_quality.py


## `01_download_official_data.py`

Downloads the official Copenhagen datasets and records where each file came from.

In [5]:
%%writefile ../src/01_download_official_data.py
from __future__ import annotations

from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

from config import (
    CKAN_API_BASE,
    CKAN_ORGANIZATION,
    OFFICIAL_DATASETS,
    RAW_OFFICIAL_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import safe_write_csv, slugify, utc_now_iso


FORMAT_PRIORITY = {"geojson": 0, "shp": 1, "csv": 2}


def ckan_action(action: str, **params) -> dict:
    response = requests.get(f"{CKAN_API_BASE}/{action}", params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if not payload.get("success"):
        raise RuntimeError(f"CKAN action {action} failed: {payload}")
    return payload["result"]


def package_is_copenhagen(package: dict) -> bool:
    organization = package.get("organization") or {}
    return organization.get("name") == CKAN_ORGANIZATION


def find_package(dataset_key: str, spec: dict) -> dict:
    for package_id in spec["package_ids"]:
        try:
            package = ckan_action("package_show", id=package_id)
        except requests.HTTPError:
            continue
        if package_is_copenhagen(package):
            return package

    result = ckan_action("package_search", q=spec["search_query"], rows=10)
    for package in result.get("results", []):
        if package_is_copenhagen(package):
            return package

    raise RuntimeError(f"Could not find a Copenhagen CKAN package for {dataset_key}.")


def normalize_resource_format(resource: dict) -> str | None:
    fmt = str(resource.get("format") or "").lower()
    name = str(resource.get("name") or "").lower()
    url = str(resource.get("url") or "").lower()

    if "geojson" in fmt or name.endswith(".geojson") or "outputformat=json" in url:
        return "geojson"
    if fmt in {"shp", "shape", "shapefile"} or name.endswith(".zip") or "outputformat=shape-zip" in url:
        return "shp"
    if "csv" in fmt or name.endswith(".csv") or "outputformat=csv" in url:
        return "csv"
    return None


def choose_resource(package: dict) -> tuple[dict, str]:
    candidates = []
    for resource in package.get("resources", []):
        normalized = normalize_resource_format(resource)
        if normalized in FORMAT_PRIORITY and resource.get("url"):
            candidates.append((FORMAT_PRIORITY[normalized], resource, normalized))

    if not candidates:
        raise RuntimeError(f"No GeoJSON, SHP, or CSV resource found for {package.get('name')}.")

    _, resource, normalized = sorted(candidates, key=lambda item: item[0])[0]
    return resource, normalized


def resource_extension(resource: dict, normalized_format: str) -> str:
    name = str(resource.get("name") or "")
    suffix = Path(urlparse(name).path).suffix.lower()
    if suffix in {".geojson", ".json", ".csv", ".zip"}:
        return ".geojson" if suffix == ".json" and normalized_format == "geojson" else suffix
    return { "geojson": ".geojson", "shp": ".zip", "csv": ".csv" }[normalized_format]


def download_resource(url: str, destination: Path) -> None:
    headers = {"User-Agent": "copenhagen-osm-accessibility-analysis/1.0"}
    with requests.get(url, stream=True, timeout=180, headers=headers) as response:
        response.raise_for_status()
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("wb") as file_obj:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file_obj.write(chunk)


def main() -> None:
    ensure_directories()
    rows = []

    for dataset_key, spec in OFFICIAL_DATASETS.items():
        package = find_package(dataset_key, spec)
        resource, normalized_format = choose_resource(package)
        extension = resource_extension(resource, normalized_format)
        file_name = f"{dataset_key}_{slugify(resource.get('name'), dataset_key)}{extension}"
        destination = RAW_OFFICIAL_DIR / file_name

        print(f"Downloading {dataset_key}: {resource.get('name')} -> {destination}")
        download_resource(resource["url"], destination)

        rows.append(
            {
                "dataset_key": dataset_key,
                "dataset_name": package.get("title") or spec["dataset_name"],
                "package_id": package.get("name"),
                "resource_id": resource.get("id"),
                "resource_name": resource.get("name"),
                "source_url": resource.get("url"),
                "package_url": f"https://www.opendata.dk/city-of-copenhagen/{package.get('name')}",
                "downloaded_file": str(destination),
                "format": normalized_format,
                "download_date": utc_now_iso(),
                "license": package.get("license_title") or package.get("license_id") or "",
            }
        )

    metadata = pd.DataFrame(rows)
    safe_write_csv(metadata, RAW_OFFICIAL_DIR / "official_download_metadata.csv")
    safe_write_csv(metadata, TABLES_DIR / "official_download_metadata.csv")
    print(f"Saved metadata for {len(metadata)} official datasets.")


if __name__ == "__main__":
    main()


Overwriting ../src/01_download_official_data.py


## `03_clean_official_amenities.py`

Inspects the official layers, clips them to the municipality, and writes a shared amenity schema.

In [6]:
%%writefile ../src/03_clean_official_amenities.py
from __future__ import annotations

import pandas as pd

from config import (
    AMENITY_LAYER_KEYS,
    BYDELE_FILE,
    BOUNDARY_FILE,
    CRS_METRIC,
    CRS_WGS84,
    OFFICIAL_CLEAN_FILES,
    OFFICIAL_DATASETS,
    PROCESSED_DIR,
    RAW_OFFICIAL_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import (
    clip_to_boundary,
    dissolve_to_boundary,
    get_downloaded_resource,
    inspect_gdf,
    make_valid_geometries,
    read_metadata_table,
    read_vector_any,
    safe_write_csv,
    safe_write_gdf,
    standardize_amenities,
)


METADATA_FILE = RAW_OFFICIAL_DIR / "official_download_metadata.csv"


def load_raw_official(dataset_key: str, metadata: pd.DataFrame):
    path = get_downloaded_resource(metadata, dataset_key)
    data_format = metadata.loc[metadata["dataset_key"] == dataset_key, "format"].iloc[0]
    return read_vector_any(path, data_format=data_format)


def clean_bydele(metadata: pd.DataFrame):
    bydele = load_raw_official("bydele", metadata)
    print("Bydele columns:", list(bydele.columns))
    bydele = make_valid_geometries(bydele).to_crs(CRS_WGS84)
    safe_write_gdf(bydele, PROCESSED_DIR / "bydele_wgs84.gpkg", layer="bydele")

    bydele_metric = bydele.to_crs(CRS_METRIC)
    safe_write_gdf(bydele_metric, BYDELE_FILE, layer="bydele")

    boundary = dissolve_to_boundary(bydele_metric)
    safe_write_gdf(boundary, BOUNDARY_FILE, layer="copenhagen_boundary")
    safe_write_gdf(boundary.to_crs(CRS_WGS84), PROCESSED_DIR / "copenhagen_boundary_wgs84.gpkg", layer="copenhagen_boundary")
    return bydele, boundary


def clean_amenity_layer(dataset_key: str, metadata: pd.DataFrame, boundary) -> dict:
    spec = OFFICIAL_DATASETS[dataset_key]
    raw = load_raw_official(dataset_key, metadata)
    print(f"{dataset_key} columns:", list(raw.columns))

    inspection = inspect_gdf(spec["dataset_name"], raw)
    raw = make_valid_geometries(raw).to_crs(CRS_WGS84)
    clipped = clip_to_boundary(raw, boundary).to_crs(CRS_METRIC)

    original_path = PROCESSED_DIR / f"official_{dataset_key}_original_geometries.gpkg"
    safe_write_gdf(clipped.to_crs(CRS_WGS84), original_path, layer=f"official_{dataset_key}_original")

    standardized = standardize_amenities(
        clipped,
        amenity_type=spec["amenity_type"],
        source="official",
        dataset_label=spec["dataset_name"],
        id_prefix=f"official_{spec['amenity_type']}",
    )
    safe_write_gdf(standardized, OFFICIAL_CLEAN_FILES[dataset_key], layer=f"official_{dataset_key}")

    inspection["cleaned_row_count"] = len(standardized)
    inspection["cleaned_file"] = str(OFFICIAL_CLEAN_FILES[dataset_key])
    inspection["original_geometry_file"] = str(original_path)
    return inspection


def main() -> None:
    ensure_directories()
    metadata = read_metadata_table(METADATA_FILE)

    inspection_rows = []
    bydele_raw = load_raw_official("bydele", metadata)
    inspection_rows.append(inspect_gdf("Bydele", bydele_raw))
    _, boundary = clean_bydele(metadata)

    for dataset_key in AMENITY_LAYER_KEYS:
        inspection_rows.append(clean_amenity_layer(dataset_key, metadata, boundary))

    inspection = pd.DataFrame(inspection_rows)
    safe_write_csv(inspection, TABLES_DIR / "official_dataset_inspection.csv")
    print(f"Saved official inspection table with {len(inspection)} rows.")


if __name__ == "__main__":
    main()


Overwriting ../src/03_clean_official_amenities.py


## `02_extract_osm_data.py`

Reads the local Denmark PBF, clips to Copenhagen, and extracts the OSM amenities and walking network.

In [7]:
%%writefile ../src/02_extract_osm_data.py
from __future__ import annotations

from config import (
    AMENITY_LAYER_KEYS,
    CRS_WGS84,
    OSM_RAW_FILES,
    OSM_WALKING_EDGES_FILE,
    OSM_WALKING_NODES_FILE,
    PBF_PATH,
    ensure_directories,
)
from utils import (
    clip_to_boundary,
    ensure_crs,
    import_geopandas,
    load_boundary,
    make_valid_geometries,
    require_file,
    resolve_edge_endpoint_columns,
    resolve_node_id_column,
    safe_write_gdf,
)


OSM_AMENITY_FILTERS = {
    "libraries": {"amenity": ["library"]},
    "playgrounds": {"leisure": ["playground"]},
    "sports_facilities": {"leisure": ["sports_centre", "sports_hall"]},
}


def import_pyrosm():
    try:
        from pyrosm import OSM
    except ImportError as exc:
        raise ImportError(
            "OSM extraction requires pyrosm. Install it with the conda-forge environment "
            "recommended in README.md. The script will not use Overpass or download a PBF."
        ) from exc
    return OSM


def load_osm_reader(boundary_wgs84):
    OSM = import_pyrosm()
    minx, miny, maxx, maxy = boundary_wgs84.total_bounds
    return OSM(str(PBF_PATH), bounding_box=[minx, miny, maxx, maxy])


def extract_amenities(osm, boundary_wgs84) -> None:
    gpd = import_geopandas()
    for layer_key in AMENITY_LAYER_KEYS:
        custom_filter = OSM_AMENITY_FILTERS[layer_key]
        print(f"Extracting OSM {layer_key}: {custom_filter}")
        data = osm.get_data_by_custom_criteria(
            custom_filter=custom_filter,
            filter_type="keep",
            keep_nodes=True,
            keep_ways=True,
            keep_relations=True,
        )
        if data is None or len(data) == 0:
            data = gpd.GeoDataFrame(geometry=[], crs=CRS_WGS84)
        data = ensure_crs(data, CRS_WGS84)
        data = clip_to_boundary(data, boundary_wgs84)
        safe_write_gdf(data.to_crs(CRS_WGS84), OSM_RAW_FILES[layer_key], layer=f"osm_{layer_key}")
        print(f"Saved {len(data)} OSM {layer_key} features.")


def extract_walking_network(osm, boundary_wgs84) -> None:
    print("Extracting OSM walking network from local PBF.")
    nodes, edges = osm.get_network(network_type="walking", nodes=True)
    nodes = make_valid_geometries(ensure_crs(nodes, CRS_WGS84))
    edges = make_valid_geometries(ensure_crs(edges, CRS_WGS84))

    boundary_geom = boundary_wgs84.geometry.iloc[0]
    edges = edges[edges.intersects(boundary_geom)].copy()

    try:
        u_col, v_col = resolve_edge_endpoint_columns(edges)
        node_col = resolve_node_id_column(nodes)
        used_nodes = set(edges[u_col].astype(str)) | set(edges[v_col].astype(str))
        nodes = nodes[nodes[node_col].astype(str).isin(used_nodes)].copy()
    except ValueError:
        nodes = nodes[nodes.intersects(boundary_geom)].copy()

    safe_write_gdf(edges.to_crs(CRS_WGS84), OSM_WALKING_EDGES_FILE, layer="osm_walking_edges")
    safe_write_gdf(nodes.to_crs(CRS_WGS84), OSM_WALKING_NODES_FILE, layer="osm_walking_nodes")
    print(f"Saved walking network: {len(nodes)} nodes, {len(edges)} edges.")


def main() -> None:
    ensure_directories()
    require_file(
        PBF_PATH,
        f"Local OSM PBF not found at {PBF_PATH}. Place the Denmark OSM PBF there and "
        "name it denmark-latest.osm.pbf. This project does not download the PBF.",
    )
    boundary_wgs84 = load_boundary(metric=False).to_crs(CRS_WGS84)
    osm = load_osm_reader(boundary_wgs84)
    extract_amenities(osm, boundary_wgs84)
    extract_walking_network(osm, boundary_wgs84)


if __name__ == "__main__":
    main()


Overwriting ../src/02_extract_osm_data.py


## `04_clean_osm_amenities.py`

Standardizes OSM libraries, playgrounds, and sports facilities so they can be compared with the official layers.

In [8]:
%%writefile ../src/04_clean_osm_amenities.py
from __future__ import annotations

import pandas as pd

from config import (
    AMENITY_LAYER_KEYS,
    AMENITY_TYPE_BY_LAYER,
    OSM_CLEAN_FILES,
    OSM_RAW_FILES,
    PROCESSED_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import (
    clip_to_boundary,
    import_geopandas,
    inspect_gdf,
    load_boundary,
    make_valid_geometries,
    require_file,
    safe_write_csv,
    safe_write_gdf,
    standardize_amenities,
)


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()
    boundary = load_boundary(metric=True)
    inspection_rows = []

    for layer_key in AMENITY_LAYER_KEYS:
        source_path = require_file(OSM_RAW_FILES[layer_key])
        raw = gpd.read_file(source_path)
        print(f"OSM {layer_key} columns:", list(raw.columns))

        inspection = inspect_gdf(f"osm_{layer_key}", raw)
        raw = make_valid_geometries(raw)
        clipped = clip_to_boundary(raw, boundary)

        original_path = PROCESSED_DIR / f"osm_{layer_key}_original_geometries.gpkg"
        safe_write_gdf(clipped, original_path, layer=f"osm_{layer_key}_original")

        amenity_type = AMENITY_TYPE_BY_LAYER[layer_key]
        standardized = standardize_amenities(
            clipped,
            amenity_type=amenity_type,
            source="osm",
            dataset_label=f"osm_{layer_key}",
            id_prefix=f"osm_{amenity_type}",
        )
        safe_write_gdf(standardized, OSM_CLEAN_FILES[layer_key], layer=f"osm_{layer_key}")

        inspection["cleaned_row_count"] = len(standardized)
        inspection["cleaned_file"] = str(OSM_CLEAN_FILES[layer_key])
        inspection["original_geometry_file"] = str(original_path)
        inspection_rows.append(inspection)

    safe_write_csv(pd.DataFrame(inspection_rows), TABLES_DIR / "osm_dataset_inspection.csv")
    print(f"Saved cleaned OSM amenity layers for {len(inspection_rows)} amenity types.")


if __name__ == "__main__":
    main()


Overwriting ../src/04_clean_osm_amenities.py


## `05_prepare_walking_network.py`

Turns the extracted OSM walking network into clean nodes, clean edges, and a NetworkX graph with walking times.

In [9]:
%%writefile ../src/05_prepare_walking_network.py
from __future__ import annotations

import math

import networkx as nx

from config import (
    CRS_METRIC,
    OSM_WALKING_EDGES_CLEAN_FILE,
    OSM_WALKING_EDGES_FILE,
    OSM_WALKING_GRAPH_FILE,
    OSM_WALKING_NODES_CLEAN_FILE,
    OSM_WALKING_NODES_FILE,
    WALKING_SPEED_KMH,
    ensure_directories,
)
from utils import (
    ensure_crs,
    import_geopandas,
    make_valid_geometries,
    require_file,
    resolve_edge_endpoint_columns,
    resolve_node_id_column,
    safe_write_gdf,
)


NON_WALKABLE_HIGHWAYS = {
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "construction",
    "proposed",
    "raceway",
}

BLOCKING_VALUES = {"no", "private", "customers"}


def lower_text(value) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return str(value).lower()


def filter_walkable_edges(edges):
    keep = edges.geometry.notna() & ~edges.geometry.is_empty
    if "highway" in edges.columns:
        keep &= ~edges["highway"].apply(lambda value: lower_text(value) in NON_WALKABLE_HIGHWAYS)
    if "access" in edges.columns:
        keep &= ~edges["access"].apply(lambda value: lower_text(value) in BLOCKING_VALUES)
    if "foot" in edges.columns:
        keep &= ~edges["foot"].apply(lambda value: lower_text(value) in {"no", "private"})
    return edges[keep].copy()


def prepare_nodes(nodes):
    nodes = make_valid_geometries(ensure_crs(nodes)).to_crs(CRS_METRIC)
    node_col = resolve_node_id_column(nodes)
    nodes = nodes.copy()
    nodes["node_id"] = nodes[node_col].astype(str)
    nodes["x"] = nodes.geometry.x
    nodes["y"] = nodes.geometry.y
    return nodes


def prepare_edges(edges):
    edges = make_valid_geometries(ensure_crs(edges)).to_crs(CRS_METRIC)
    edges = filter_walkable_edges(edges)
    u_col, v_col = resolve_edge_endpoint_columns(edges)
    edges = edges.copy()
    edges["u"] = edges[u_col].astype(str)
    edges["v"] = edges[v_col].astype(str)
    edges["length_m"] = edges.geometry.length
    speed_m_per_min = WALKING_SPEED_KMH * 1000.0 / 60.0
    edges["walking_time_minutes"] = edges["length_m"] / speed_m_per_min
    return edges


def build_graph(nodes, edges) -> nx.Graph:
    graph = nx.Graph()

    for row in nodes.itertuples(index=False):
        node_id = str(row.node_id)
        graph.add_node(node_id, x=float(row.x), y=float(row.y))

    for row in edges.itertuples(index=False):
        u = str(row.u)
        v = str(row.v)
        if u not in graph or v not in graph or u == v:
            continue
        attrs = {
            "length_m": float(row.length_m),
            "walking_time_minutes": float(row.walking_time_minutes),
        }
        if hasattr(row, "highway"):
            attrs["highway"] = str(getattr(row, "highway"))
        if hasattr(row, "name"):
            attrs["name"] = str(getattr(row, "name"))

        if graph.has_edge(u, v):
            if attrs["walking_time_minutes"] < graph[u][v].get("walking_time_minutes", float("inf")):
                graph[u][v].update(attrs)
        else:
            graph.add_edge(u, v, **attrs)

    return graph


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()

    raw_edges = gpd.read_file(require_file(OSM_WALKING_EDGES_FILE))
    raw_nodes = gpd.read_file(require_file(OSM_WALKING_NODES_FILE))

    nodes = prepare_nodes(raw_nodes)
    edges = prepare_edges(raw_edges)
    used_nodes = set(edges["u"]) | set(edges["v"])
    nodes = nodes[nodes["node_id"].isin(used_nodes)].copy()
    edges = edges[edges["u"].isin(set(nodes["node_id"])) & edges["v"].isin(set(nodes["node_id"]))].copy()

    graph = build_graph(nodes, edges)
    safe_write_gdf(edges, OSM_WALKING_EDGES_CLEAN_FILE, layer="osm_walking_edges")
    safe_write_gdf(nodes, OSM_WALKING_NODES_CLEAN_FILE, layer="osm_walking_nodes")

    try:
        nx.write_graphml(graph, OSM_WALKING_GRAPH_FILE)
        print(f"Saved GraphML: {OSM_WALKING_GRAPH_FILE}")
    except Exception as exc:
        print(f"Could not save GraphML ({exc}). Clean edge/node GeoPackages were still saved.")

    print(
        "Prepared walking network: "
        f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} undirected edges."
    )


if __name__ == "__main__":
    main()


Overwriting ../src/05_prepare_walking_network.py


## `06_create_origin_grid.py`

Builds the 500 m grid and centroid origins over Copenhagen Municipality.

In [10]:
%%writefile ../src/06_create_origin_grid.py
from __future__ import annotations

import numpy as np

from config import (
    CRS_WGS84,
    CRS_METRIC,
    GRID_SIZE_M,
    ORIGINS_GRID_FILE,
    ORIGINS_POINTS_FILE,
    ensure_directories,
)
from utils import import_geopandas, import_shapely_geometry, load_boundary, safe_write_gdf


def create_grid(boundary, cell_size_m: int):
    gpd = import_geopandas()
    _, box, _, _ = import_shapely_geometry()

    boundary = boundary.to_crs(CRS_METRIC)
    minx, miny, maxx, maxy = boundary.total_bounds
    x_coords = np.arange(np.floor(minx / cell_size_m) * cell_size_m, maxx + cell_size_m, cell_size_m)
    y_coords = np.arange(np.floor(miny / cell_size_m) * cell_size_m, maxy + cell_size_m, cell_size_m)

    cells = []
    for x in x_coords:
        for y in y_coords:
            cells.append(box(x, y, x + cell_size_m, y + cell_size_m))

    grid = gpd.GeoDataFrame({"cell_id": range(1, len(cells) + 1)}, geometry=cells, crs=CRS_METRIC)
    mask = grid.intersects(boundary.geometry.iloc[0])
    grid = grid[mask].copy().reset_index(drop=True)
    grid["origin_id"] = [f"origin_{i + 1:05d}" for i in range(len(grid))]
    grid["grid_size_m"] = cell_size_m
    return grid[["origin_id", "grid_size_m", "geometry"]]


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()
    boundary = load_boundary(metric=True)
    grid = create_grid(boundary, GRID_SIZE_M)
    centroids_metric = grid.geometry.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_metric, crs=CRS_METRIC).to_crs(CRS_WGS84)

    points = gpd.GeoDataFrame(
        {
            "origin_id": grid["origin_id"],
            "grid_size_m": grid["grid_size_m"],
            "longitude": centroids_wgs84.x,
            "latitude": centroids_wgs84.y,
        },
        geometry=centroids_metric,
        crs=CRS_METRIC,
    )

    safe_write_gdf(grid, ORIGINS_GRID_FILE, layer="origins_grid_500m")
    safe_write_gdf(points, ORIGINS_POINTS_FILE, layer="origins_points_500m")
    print(f"Saved {len(grid)} origin grid cells and centroid points.")


if __name__ == "__main__":
    main()


Overwriting ../src/06_create_origin_grid.py


## `07_match_osm_to_official.py`

Matches OSM amenities to nearby official amenities and writes the completeness diagnostics.

In [11]:
%%writefile ../src/07_match_osm_to_official.py
from __future__ import annotations

import math

import pandas as pd

from config import (
    AMENITY_LAYER_KEYS,
    AMENITY_TYPE_BY_LAYER,
    CRS_METRIC,
    CRS_WGS84,
    MATCH_THRESHOLDS_M,
    OSM_CLEAN_FILES,
    OFFICIAL_CLEAN_FILES,
    MAPS_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import import_geopandas, safe_write_csv, safe_write_gdf


def one_to_one_nearest_matches(official, osm, threshold_m: float) -> list[dict]:
    try:
        from scipy.spatial import cKDTree
    except ImportError as exc:
        raise ImportError("POI matching requires scipy. Install dependencies from requirements.txt.") from exc

    official = official.to_crs(CRS_METRIC).reset_index(drop=True)
    osm = osm.to_crs(CRS_METRIC).reset_index(drop=True)
    if official.empty or osm.empty:
        return []

    official_coords = list(zip(official.geometry.x, official.geometry.y))
    osm_coords = list(zip(osm.geometry.x, osm.geometry.y))
    tree = cKDTree(osm_coords)

    candidate_pairs = []
    for official_idx, coord in enumerate(official_coords):
        osm_indices = tree.query_ball_point(coord, r=threshold_m)
        ox, oy = coord
        for osm_idx in osm_indices:
            sx, sy = osm_coords[osm_idx]
            distance = math.hypot(ox - sx, oy - sy)
            candidate_pairs.append((distance, official_idx, osm_idx))

    matched_official = set()
    matched_osm = set()
    matches = []
    for distance, official_idx, osm_idx in sorted(candidate_pairs, key=lambda item: item[0]):
        if official_idx in matched_official or osm_idx in matched_osm:
            continue
        matched_official.add(official_idx)
        matched_osm.add(osm_idx)
        matches.append(
            {
                "official_index": official_idx,
                "osm_index": osm_idx,
                "match_distance_m": distance,
                "official_amenity_id": official.iloc[official_idx]["amenity_id"],
                "osm_amenity_id": osm.iloc[osm_idx]["amenity_id"],
            }
        )
    return matches


def build_match_status_layer(official, osm, matches, amenity_type: str):
    gpd = import_geopandas()
    official = official.to_crs(CRS_WGS84).reset_index(drop=True)
    osm = osm.to_crs(CRS_WGS84).reset_index(drop=True)
    official_matched = {match["official_index"] for match in matches}
    osm_matched = {match["osm_index"] for match in matches}

    official_status = official.copy()
    official_status["match_status"] = [
        "matched_official" if idx in official_matched else "unmatched_official"
        for idx in range(len(official_status))
    ]

    osm_status = osm.copy()
    osm_status["match_status"] = [
        "matched_osm" if idx in osm_matched else "unmatched_osm"
        for idx in range(len(osm_status))
    ]

    combined = pd.concat([official_status, osm_status], ignore_index=True)
    combined["amenity_type"] = amenity_type
    return gpd.GeoDataFrame(combined, geometry="geometry", crs=CRS_WGS84)


def make_interactive_poi_map(status_layers) -> None:
    try:
        import folium
    except ImportError:
        print("folium is not installed; skipping matched_unmatched_pois.html.")
        return

    if not status_layers:
        return

    all_points = pd.concat(status_layers, ignore_index=True)
    if all_points.empty:
        return

    center = [all_points.geometry.y.mean(), all_points.geometry.x.mean()]
    fmap = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
    colors = {
        "matched_official": "#1b9e77",
        "matched_osm": "#66a61e",
        "unmatched_official": "#d95f02",
        "unmatched_osm": "#7570b3",
    }

    for _, row in all_points.iterrows():
        tooltip = (
            f"{row.get('amenity_type', '')} | {row.get('source', '')} | "
            f"{row.get('match_status', '')} | {row.get('name', '')}"
        )
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4,
            color=colors.get(row["match_status"], "#333333"),
            fill=True,
            fill_opacity=0.75,
            tooltip=tooltip,
        ).add_to(fmap)

    output = MAPS_DIR / "matched_unmatched_pois.html"
    output.parent.mkdir(parents=True, exist_ok=True)
    fmap.save(output)
    print(f"Saved {output}")


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()
    summary_rows = []
    detail_rows = []
    status_layers = []

    for layer_key in AMENITY_LAYER_KEYS:
        amenity_type = AMENITY_TYPE_BY_LAYER[layer_key]
        official = gpd.read_file(OFFICIAL_CLEAN_FILES[layer_key]).to_crs(CRS_METRIC)
        osm = gpd.read_file(OSM_CLEAN_FILES[layer_key]).to_crs(CRS_METRIC)
        threshold = MATCH_THRESHOLDS_M[amenity_type]

        matches = one_to_one_nearest_matches(official, osm, threshold)
        for match in matches:
            match["amenity_type"] = amenity_type
            match["threshold_m"] = threshold
        detail_rows.extend(matches)

        matched_count = len(matches)
        official_count = len(official)
        osm_count = len(osm)
        distances = [match["match_distance_m"] for match in matches]
        summary_rows.append(
            {
                "amenity_type": amenity_type,
                "threshold_m": threshold,
                "official_count": official_count,
                "osm_count": osm_count,
                "matched_count": matched_count,
                "unmatched_official_count": official_count - matched_count,
                "unmatched_osm_count": osm_count - matched_count,
                "recall": matched_count / official_count if official_count else None,
                "precision": matched_count / osm_count if osm_count else None,
                "median_match_distance": pd.Series(distances).median() if distances else None,
            }
        )

        status_layer = build_match_status_layer(official, osm, matches, amenity_type)
        status_layers.append(status_layer)

    safe_write_csv(pd.DataFrame(summary_rows), TABLES_DIR / "poi_completeness_summary.csv")
    safe_write_csv(pd.DataFrame(detail_rows), TABLES_DIR / "poi_match_details.csv")

    if status_layers:
        combined_status = pd.concat(status_layers, ignore_index=True)
        combined_status = gpd.GeoDataFrame(combined_status, geometry="geometry", crs=CRS_WGS84)
        safe_write_gdf(combined_status, MAPS_DIR / "matched_unmatched_pois.gpkg", layer="matched_unmatched_pois")
        make_interactive_poi_map(status_layers)

    print("Saved POI completeness summary and match details.")


if __name__ == "__main__":
    main()


Overwriting ../src/07_match_osm_to_official.py


## `08_compute_accessibility.py`

Runs the baseline 15-minute walking accessibility calculation.

In [12]:
%%writefile ../src/08_compute_accessibility.py
from __future__ import annotations

import math

import networkx as nx
import pandas as pd

from config import (
    ACCESSIBILITY_GPKG,
    ACCESSIBILITY_TABLE,
    ACCESS_THRESHOLD_MINUTES,
    AMENITY_LAYER_KEYS,
    AMENITY_TYPE_BY_LAYER,
    CRS_METRIC,
    OFFICIAL_CLEAN_FILES,
    ORIGINS_POINTS_FILE,
    ORIGINS_POINTS_SNAPPED_FILE,
    OSM_CLEAN_FILES,
    OSM_WALKING_EDGES_CLEAN_FILE,
    OSM_WALKING_GRAPH_FILE,
    OSM_WALKING_NODES_CLEAN_FILE,
    PROCESSED_DIR,
    SNAP_WARNING_DISTANCE_M,
    TABLES_DIR,
    ensure_directories,
)
from utils import (
    clean_numeric_difference,
    import_geopandas,
    nearest_node_snap,
    require_file,
    resolve_node_id_column,
    safe_write_csv,
    safe_write_gdf,
)


def coerce_graph_weights(graph: nx.Graph) -> nx.Graph:
    for _, _, data in graph.edges(data=True):
        for key in ["length_m", "walking_time_minutes"]:
            if key in data:
                data[key] = float(data[key])
    return graph


def load_graph() -> nx.Graph:
    if OSM_WALKING_GRAPH_FILE.exists():
        return coerce_graph_weights(nx.read_graphml(OSM_WALKING_GRAPH_FILE))

    gpd = import_geopandas()
    nodes = gpd.read_file(require_file(OSM_WALKING_NODES_CLEAN_FILE)).to_crs(CRS_METRIC)
    edges = gpd.read_file(require_file(OSM_WALKING_EDGES_CLEAN_FILE)).to_crs(CRS_METRIC)
    graph = nx.Graph()
    for row in nodes.itertuples(index=False):
        graph.add_node(str(row.node_id), x=float(row.geometry.x), y=float(row.geometry.y))
    for row in edges.itertuples(index=False):
        if not hasattr(row, "u") or not hasattr(row, "v"):
            raise ValueError("Clean walking edges must include u and v columns.")
        graph.add_edge(
            str(row.u),
            str(row.v),
            length_m=float(row.length_m),
            walking_time_minutes=float(row.walking_time_minutes),
        )
    return graph


def add_snap_columns(points, nodes, point_id_col: str):
    snap = nearest_node_snap(points, nodes, point_id_col=point_id_col)
    out = points.merge(snap, on=point_id_col, how="left")
    out["snap_flag_gt_100m"] = out["snap_distance_m"] > SNAP_WARNING_DISTANCE_M
    out["nearest_node"] = out["nearest_node"].astype(str)
    return out


def snapping_summary(label: str, source: str, amenity_type: str | None, snapped) -> dict:
    distances = pd.to_numeric(snapped["snap_distance_m"], errors="coerce")
    return {
        "layer": label,
        "source": source,
        "amenity_type": amenity_type or "",
        "record_count": len(snapped),
        "flagged_gt_100m_count": int((distances > SNAP_WARNING_DISTANCE_M).sum()),
        "flagged_gt_100m_share": float((distances > SNAP_WARNING_DISTANCE_M).mean()) if len(snapped) else None,
        "median_snap_distance_m": float(distances.median()) if len(snapped) else None,
        "max_snap_distance_m": float(distances.max()) if len(snapped) else None,
    }


def shortest_times_to_sources(graph: nx.Graph, source_nodes: set[str]) -> dict[str, float]:
    source_nodes = {str(node) for node in source_nodes if str(node) in graph}
    if not source_nodes:
        return {}
    return nx.multi_source_dijkstra_path_length(
        graph,
        sources=source_nodes,
        weight="walking_time_minutes",
    )


def finite_access(value: float) -> bool:
    return math.isfinite(value) and value <= ACCESS_THRESHOLD_MINUTES


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()
    graph = load_graph()

    nodes = gpd.read_file(require_file(OSM_WALKING_NODES_CLEAN_FILE)).to_crs(CRS_METRIC)
    node_col = resolve_node_id_column(nodes)
    nodes[node_col] = nodes[node_col].astype(str)

    origins = gpd.read_file(require_file(ORIGINS_POINTS_FILE)).to_crs(CRS_METRIC)
    origins_snapped = add_snap_columns(origins, nodes, "origin_id")
    safe_write_gdf(origins_snapped, ORIGINS_POINTS_SNAPPED_FILE, layer="origins_points_500m")

    diagnostics = [snapping_summary("origins_points_500m", "origin", None, origins_snapped)]
    snapped_layers = {"official": {}, "osm": {}}

    for layer_key in AMENITY_LAYER_KEYS:
        amenity_type = AMENITY_TYPE_BY_LAYER[layer_key]
        for source, files in [("official", OFFICIAL_CLEAN_FILES), ("osm", OSM_CLEAN_FILES)]:
            amenities = gpd.read_file(require_file(files[layer_key])).to_crs(CRS_METRIC)
            snapped = add_snap_columns(amenities, nodes, "amenity_id")
            output = PROCESSED_DIR / f"{source}_{layer_key}_snapped.gpkg"
            safe_write_gdf(snapped, output, layer=f"{source}_{layer_key}_snapped")
            diagnostics.append(snapping_summary(f"{source}_{layer_key}", source, amenity_type, snapped))
            snapped_layers[source][amenity_type] = snapped

    safe_write_csv(pd.DataFrame(diagnostics), TABLES_DIR / "snapping_diagnostics.csv")

    origin_nodes = origins_snapped.set_index("origin_id")["nearest_node"].astype(str).to_dict()
    rows = []

    for amenity_type in AMENITY_TYPE_BY_LAYER.values():
        official_nodes = set(snapped_layers["official"][amenity_type]["nearest_node"].dropna().astype(str))
        osm_nodes = set(snapped_layers["osm"][amenity_type]["nearest_node"].dropna().astype(str))
        official_lengths = shortest_times_to_sources(graph, official_nodes)
        osm_lengths = shortest_times_to_sources(graph, osm_nodes)

        for origin_id, origin_node in origin_nodes.items():
            official_time = official_lengths.get(origin_node, float("inf"))
            osm_time = osm_lengths.get(origin_node, float("inf"))
            official_access = finite_access(official_time)
            osm_access = finite_access(osm_time)
            rows.append(
                {
                    "origin_id": origin_id,
                    "amenity_type": amenity_type,
                    "official_nearest_time": official_time if math.isfinite(official_time) else None,
                    "osm_nearest_time": osm_time if math.isfinite(osm_time) else None,
                    "official_access_15": official_access,
                    "osm_access_15": osm_access,
                    "difference_minutes": clean_numeric_difference(osm_time, official_time),
                    "origin_nearest_node": origin_node,
                }
            )

    results = pd.DataFrame(rows)
    safe_write_csv(results, ACCESSIBILITY_TABLE)

    result_gdf = results.merge(
        origins_snapped[["origin_id", "snap_distance_m", "snap_flag_gt_100m", "geometry"]],
        on="origin_id",
        how="left",
    )
    result_gdf = gpd.GeoDataFrame(result_gdf, geometry="geometry", crs=CRS_METRIC)
    safe_write_gdf(result_gdf, ACCESSIBILITY_GPKG, layer="origin_accessibility_comparison")

    print(f"Saved accessibility comparison for {len(results)} origin/category rows.")


if __name__ == "__main__":
    main()


Overwriting ../src/08_compute_accessibility.py


## `09_compare_accessibility.py`

Classifies agreement and disagreement between the OSM and official accessibility results.

In [13]:
%%writefile ../src/09_compare_accessibility.py
from __future__ import annotations

import argparse

import pandas as pd

from config import (
    ACCESSIBILITY_TABLE,
    BYDELE_FILE,
    CLASSIFIED_ACCESSIBILITY_GPKG,
    CLASSIFIED_ACCESSIBILITY_TABLE,
    COMPOSITE_DISAGREEMENT_GPKG,
    COMPOSITE_DISAGREEMENT_TABLE,
    CRS_METRIC,
    ORIGINS_GRID_FILE,
    ORIGINS_POINTS_SNAPPED_FILE,
    TABLES_DIR,
    ensure_directories,
)
from utils import (
    DISTRICT_NAME_CANDIDATES,
    classify_accessibility,
    find_column,
    import_geopandas,
    require_file,
    safe_write_csv,
    safe_write_gdf,
)


DISAGREEMENT_CLASSES = {"osm_false_access", "osm_hidden_access"}


def to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def classify_rows(accessibility: pd.DataFrame) -> pd.DataFrame:
    out = accessibility.copy()
    out["official_access_15"] = out["official_access_15"].apply(to_bool)
    out["osm_access_15"] = out["osm_access_15"].apply(to_bool)
    out["distortion_class"] = [
        classify_accessibility(official, osm)
        for official, osm in zip(out["official_access_15"], out["osm_access_15"])
    ]
    out["disagrees"] = out["distortion_class"].isin(DISAGREEMENT_CLASSES)
    return out


def save_classified_geodata(classified: pd.DataFrame):
    gpd = import_geopandas()
    origins = gpd.read_file(require_file(ORIGINS_POINTS_SNAPPED_FILE)).to_crs(CRS_METRIC)
    gdf = classified.merge(origins[["origin_id", "geometry"]], on="origin_id", how="left")
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=CRS_METRIC)
    safe_write_gdf(gdf, CLASSIFIED_ACCESSIBILITY_GPKG, layer="origin_accessibility_classified")


def save_composite_disagreement(classified: pd.DataFrame):
    gpd = import_geopandas()
    composite = (
        classified.assign(disagreement=classified["distortion_class"].isin(DISAGREEMENT_CLASSES).astype(int))
        .groupby("origin_id", as_index=False)["disagreement"]
        .sum()
        .rename(columns={"disagreement": "composite_disagreement_score"})
    )
    safe_write_csv(composite, COMPOSITE_DISAGREEMENT_TABLE)

    grid = gpd.read_file(require_file(ORIGINS_GRID_FILE)).to_crs(CRS_METRIC)
    composite_gdf = grid.merge(composite, on="origin_id", how="left")
    composite_gdf["composite_disagreement_score"] = composite_gdf["composite_disagreement_score"].fillna(0).astype(int)
    safe_write_gdf(composite_gdf, COMPOSITE_DISAGREEMENT_GPKG, layer="composite_disagreement")
    return composite_gdf


def district_summary(classified: pd.DataFrame) -> pd.DataFrame:
    gpd = import_geopandas()
    origins = gpd.read_file(require_file(ORIGINS_POINTS_SNAPPED_FILE)).to_crs(CRS_METRIC)
    bydele = gpd.read_file(require_file(BYDELE_FILE)).to_crs(CRS_METRIC)
    district_col = find_column(bydele.columns, DISTRICT_NAME_CANDIDATES)
    if district_col is None:
        bydele = bydele.reset_index().rename(columns={"index": "district"})
        district_col = "district"

    districts = bydele[[district_col, "geometry"]].rename(columns={district_col: "district"})
    origin_districts = gpd.sjoin(
        origins[["origin_id", "geometry"]],
        districts,
        how="left",
        predicate="intersects",
    )[["origin_id", "district"]].drop_duplicates(subset=["origin_id"])
    data = classified.merge(origin_districts, on="origin_id", how="left")
    data["district"] = data["district"].fillna("Unassigned")
    data["difference_minutes"] = pd.to_numeric(data["difference_minutes"], errors="coerce")

    rows = []
    for (district, amenity_type), group in data.groupby(["district", "amenity_type"], dropna=False):
        official_share = group["official_access_15"].mean()
        osm_share = group["osm_access_15"].mean()
        rows.append(
            {
                "district": district,
                "amenity_type": amenity_type,
                "n_origins": int(group["origin_id"].nunique()),
                "official_share_accessible_15": official_share,
                "osm_share_accessible_15": osm_share,
                "difference_share_percentage_points": (osm_share - official_share) * 100.0,
                "share_osm_false_access": (group["distortion_class"] == "osm_false_access").mean(),
                "share_osm_hidden_access": (group["distortion_class"] == "osm_hidden_access").mean(),
                "median_time_difference": group["difference_minutes"].median(),
            }
        )
    return pd.DataFrame(rows)


def optional_spatial_autocorrelation(composite_gdf) -> None:
    try:
        from esda.moran import Moran, Moran_Local
        from libpysal.weights import Queen
    except ImportError:
        print("libpysal/esda are not installed; skipping optional Moran analysis.")
        return

    weights = Queen.from_dataframe(composite_gdf, use_index=False)
    weights.transform = "r"
    values = composite_gdf["composite_disagreement_score"].astype(float).to_numpy()
    moran = Moran(values, weights)
    local = Moran_Local(values, weights)

    summary = pd.DataFrame(
        [
            {
                "statistic": "global_moran_i",
                "value": moran.I,
                "p_sim": moran.p_sim,
                "permutations": moran.permutations,
            }
        ]
    )
    safe_write_csv(summary, TABLES_DIR / "spatial_autocorrelation_summary.csv")

    lisa = composite_gdf.copy()
    lisa["local_moran_i"] = local.Is
    lisa["local_moran_p_sim"] = local.p_sim
    lisa["local_moran_quadrant"] = local.q
    safe_write_gdf(lisa, TABLES_DIR / "local_moran_composite.gpkg", layer="local_moran_composite")
    print("Saved optional Moran and Local Moran outputs.")


def write_limitations_notes() -> None:
    text = """# Limitations notes

1. Official datasets are treated as reference data but may also be incomplete or maintained for municipal purposes.
2. OSM is volunteered geographic information and can contain missing, duplicated or differently classified amenities.
3. Sports facilities are especially sensitive to classification because OSM may map whole sports centres, halls or individual pitches differently.
4. Using the same OSM walking network isolates amenity-data distortion but does not test OSM network distortion.
5. Point/polygon conversion can affect measured walking distance.
6. A 500 m grid reduces but does not eliminate aggregation bias.
7. The 15-minute threshold depends on assumed walking speed.
8. Opening hours are not included unless available consistently in both sources.
9. Some amenities may be private, restricted or not publicly accessible even if they appear in the data.
10. Results should be interpreted as data-source sensitivity, not as absolute ground truth.

## Origin-grid and snapping quality

1. The original analysis used a 500 m grid of origins.
2. Some grid cells were located near harbour, water, parks, industrial areas or irregular municipal edges.
3. These cells can produce large snapping distances to the walking network.
4. District assignment based only on centroids can leave valid edge cells unassigned.
5. To reduce this problem, the revised analysis assigns districts by largest-area overlap and runs a sensitivity analysis excluding origins with large snapping distances.
6. Results should therefore be interpreted as grid-based accessibility estimates, not exact household-level accessibility.
"""
    output = TABLES_DIR / "limitations_notes.md"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(text, encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--moran", action="store_true", help="Run optional spatial autocorrelation analysis.")
    args = parser.parse_args()

    ensure_directories()
    accessibility = pd.read_csv(require_file(ACCESSIBILITY_TABLE))
    classified = classify_rows(accessibility)
    safe_write_csv(classified, CLASSIFIED_ACCESSIBILITY_TABLE)
    save_classified_geodata(classified)

    composite_gdf = save_composite_disagreement(classified)
    summary = district_summary(classified)
    safe_write_csv(summary, TABLES_DIR / "district_accessibility_summary.csv")
    write_limitations_notes()

    if args.moran:
        optional_spatial_autocorrelation(composite_gdf)

    print("Saved classified accessibility, composite disagreement, district summary, and limitations notes.")


if __name__ == "__main__":
    main()


Overwriting ../src/09_compare_accessibility.py


## `10_make_maps.py`

Makes the first-pass static maps and the matched/unmatched POI map.

In [14]:
%%writefile ../src/10_make_maps.py
from __future__ import annotations

import pandas as pd

from config import (
    BYDELE_FILE,
    CLASSIFIED_ACCESSIBILITY_TABLE,
    COMPOSITE_DISAGREEMENT_GPKG,
    CRS_METRIC,
    FIGURES_DIR,
    MAPS_DIR,
    ORIGINS_GRID_FILE,
    ensure_directories,
)
from utils import import_geopandas, require_file


ACCESS_COLORS = {
    True: "#2ca25f",
    False: "#f0f0f0",
}

DISAGREEMENT_COLORS = {
    "agreement_accessible": "#2ca25f",
    "agreement_inaccessible": "#d9d9d9",
    "osm_false_access": "#d73027",
    "osm_hidden_access": "#4575b4",
}

POI_STATUS_COLORS = {
    "matched_official": "#1b9e77",
    "matched_osm": "#66a61e",
    "unmatched_official": "#d95f02",
    "unmatched_osm": "#7570b3",
}


def setup_matplotlib():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def add_basemap(ax, crs) -> None:
    try:
        import contextily as ctx
    except ImportError:
        return
    try:
        ctx.add_basemap(ax, crs=crs, source=ctx.providers.CartoDB.PositronNoLabels, attribution_size=6)
    except Exception:
        return


def finish_map(fig, ax, output, title: str) -> None:
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()
    fig.tight_layout()
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=220, bbox_inches="tight")
    import matplotlib.pyplot as plt

    plt.close(fig)


def plot_access_map(grid, boundary, amenity_type: str, source: str, column: str) -> None:
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    for value, color in ACCESS_COLORS.items():
        subset = grid[grid[column] == value]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.15, label="accessible" if value else "not accessible")
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    ax.legend(loc="lower left", frameon=True)
    add_basemap(ax, grid.crs)
    output = FIGURES_DIR / f"{source}_{amenity_type}_access_15min.png"
    finish_map(fig, ax, output, f"{source.title()} 15-minute access: {amenity_type}")


def plot_disagreement_map(grid, boundary, amenity_type: str) -> None:
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    for status, color in DISAGREEMENT_COLORS.items():
        subset = grid[grid["distortion_class"] == status]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.15, label=status)
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    ax.legend(loc="lower left", frameon=True, fontsize=8)
    add_basemap(ax, grid.crs)
    output = FIGURES_DIR / f"osm_vs_official_{amenity_type}_disagreement.png"
    finish_map(fig, ax, output, f"OSM vs official disagreement: {amenity_type}")


def plot_difference_map(grid, boundary, amenity_type: str) -> None:
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    grid.plot(
        ax=ax,
        column="difference_minutes",
        cmap="RdBu_r",
        legend=True,
        edgecolor="white",
        linewidth=0.15,
        missing_kwds={"color": "#efefef", "label": "no route"},
    )
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    add_basemap(ax, grid.crs)
    output = FIGURES_DIR / f"osm_minus_official_{amenity_type}_walking_time_difference.png"
    finish_map(fig, ax, output, f"Walking-time difference, OSM minus official: {amenity_type}")


def plot_composite_map(boundary) -> None:
    gpd = import_geopandas()
    composite = gpd.read_file(require_file(COMPOSITE_DISAGREEMENT_GPKG)).to_crs(CRS_METRIC)
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    composite.plot(
        ax=ax,
        column="composite_disagreement_score",
        cmap="YlOrRd",
        vmin=0,
        vmax=3,
        legend=True,
        edgecolor="white",
        linewidth=0.15,
    )
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    add_basemap(ax, composite.crs)
    finish_map(fig, ax, FIGURES_DIR / "composite_disagreement_score.png", "Composite disagreement score")


def plot_matched_unmatched_pois(boundary) -> None:
    gpd = import_geopandas()
    path = MAPS_DIR / "matched_unmatched_pois.gpkg"
    if not path.exists():
        print("Matched/unmatched POI GeoPackage not found; skipping POI PNG map.")
        return
    pois = gpd.read_file(path).to_crs(CRS_METRIC)
    if pois.empty:
        return
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    for status, color in POI_STATUS_COLORS.items():
        subset = pois[pois["match_status"] == status]
        if not subset.empty:
            subset.plot(ax=ax, color=color, markersize=12, label=status, alpha=0.85)
    ax.legend(loc="lower left", frameon=True, fontsize=8)
    add_basemap(ax, pois.crs)
    finish_map(fig, ax, FIGURES_DIR / "matched_unmatched_pois.png", "Matched and unmatched POIs")


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()
    classified = pd.read_csv(require_file(CLASSIFIED_ACCESSIBILITY_TABLE))
    grid = gpd.read_file(require_file(ORIGINS_GRID_FILE)).to_crs(CRS_METRIC)
    boundary = gpd.read_file(require_file(BYDELE_FILE)).to_crs(CRS_METRIC)

    for column in ["official_access_15", "osm_access_15"]:
        classified[column] = classified[column].astype(str).str.lower().isin({"true", "1", "yes"})

    for amenity_type, group in classified.groupby("amenity_type"):
        mapped = grid.merge(group, on="origin_id", how="left")
        mapped = gpd.GeoDataFrame(mapped, geometry="geometry", crs=CRS_METRIC)
        plot_access_map(mapped, boundary, amenity_type, "official", "official_access_15")
        plot_access_map(mapped, boundary, amenity_type, "osm", "osm_access_15")
        plot_disagreement_map(mapped, boundary, amenity_type)
        plot_difference_map(mapped, boundary, amenity_type)

    plot_matched_unmatched_pois(boundary)
    plot_composite_map(boundary)
    print(f"Saved static PNG maps in {FIGURES_DIR}.")


if __name__ == "__main__":
    main()


Overwriting ../src/10_make_maps.py


## `11_diagnose_origin_quality.py`

Diagnoses why the baseline district summary produced too many unassigned origins.

In [15]:
%%writefile ../src/11_diagnose_origin_quality.py
from __future__ import annotations

import pandas as pd

from config import (
    BYDELE_FILE,
    CLASSIFIED_ACCESSIBILITY_TABLE,
    CRS_METRIC,
    ORIGINS_GRID_FILE,
    ORIGINS_POINTS_FILE,
    TABLES_DIR,
    ensure_directories,
)
from origin_quality import (
    calculate_district_overlaps,
    dissolved_boundary,
    load_districts,
    load_existing_origin_snaps,
)
from utils import import_geopandas, require_file, safe_write_csv


def load_optional_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def inference_basis(row) -> str:
    reasons = []
    if row["municipality_overlap_share"] < 0.95:
        reasons.append("partial municipality overlap")
    if row["overlap_share"] < 0.10:
        reasons.append("low district overlap")
    if not bool(row["centroid_within_any_district"]):
        reasons.append("centroid outside dissolved district boundary")
    return "; ".join(reasons) if reasons else "no edge inference"


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()

    grid = gpd.read_file(require_file(ORIGINS_GRID_FILE)).to_crs(CRS_METRIC)
    points = gpd.read_file(require_file(ORIGINS_POINTS_FILE)).to_crs(CRS_METRIC)
    districts = load_districts()
    require_file(BYDELE_FILE)

    snapping_diagnostics = load_optional_csv(TABLES_DIR / "snapping_diagnostics.csv")
    accessibility = pd.read_csv(require_file(CLASSIFIED_ACCESSIBILITY_TABLE))
    district_summary = pd.read_csv(require_file(TABLES_DIR / "district_accessibility_summary.csv"))

    assignments, overlap_details = calculate_district_overlaps(grid, points, districts)
    snapped_points = load_existing_origin_snaps(points)
    if snapped_points is not None:
        assignments = assignments.merge(
            snapped_points[["origin_id", "nearest_node_id", "snap_distance_m"]],
            on="origin_id",
            how="left",
        )
    else:
        assignments["nearest_node_id"] = None
        assignments["snap_distance_m"] = None

    # The current district summary has 103 unassigned origins, which is too high
    # to treat as a minor issue. This likely happens because the origin grid was
    # created by keeping cells that intersect Copenhagen Municipality, while the
    # old district assignment used origin centroids. For edge cells, harbour
    # cells and irregular district boundaries, a grid cell can intersect the
    # municipality even if its centroid falls outside a Bydele polygon. District
    # assignment should therefore use largest-area overlap between grid cells
    # and Bydele polygons, not only centroid containment.
    unassigned = assignments[
        assignments["previous_district_assignment_if_available"].isna()
        | (assignments["previous_district_assignment_if_available"].astype(str).str.strip() == "")
    ].copy()

    boundary = dissolved_boundary(districts)
    unassigned_points = points[points["origin_id"].isin(unassigned["origin_id"])].copy()
    unassigned["point_within_copenhagen_boundary"] = unassigned_points.set_index("origin_id").geometry.within(
        boundary.geometry.iloc[0]
    ).reindex(unassigned["origin_id"]).to_numpy()
    unassigned["point_within_any_bydele_polygon"] = unassigned["centroid_within_any_district"]
    unassigned["edge_harbour_inference_basis"] = unassigned.apply(inference_basis, axis=1)

    overlap_lists = (
        overlap_details.assign(
            district_overlap=lambda df: df["district"].astype(str)
            + ":"
            + df["overlap_area_m2"].round(1).astype(str)
            + "m2"
        )
        .groupby("origin_id", as_index=False)
        .agg(
            intersecting_bydele_polygons=("district", lambda values: "|".join(sorted(map(str, values)))),
            bydele_overlap_areas=("district_overlap", lambda values: "|".join(values)),
        )
    )
    unassigned = unassigned.merge(overlap_lists, on="origin_id", how="left")

    unassigned_detail = overlap_details[overlap_details["origin_id"].isin(unassigned["origin_id"])].copy()
    unassigned_detail = unassigned_detail.sort_values(["origin_id", "overlap_area_m2"], ascending=[True, False])

    diagnostic_counts = pd.DataFrame(
        [
            {
                "metric": "origin_count",
                "value": int(points["origin_id"].nunique()),
            },
            {
                "metric": "origins_in_accessibility_table",
                "value": int(accessibility["origin_id"].nunique()),
            },
            {
                "metric": "reconstructed_previous_unassigned_origins",
                "value": int(len(unassigned)),
            },
            {
                "metric": "district_summary_unassigned_n_origins",
                "value": int(
                    district_summary.loc[
                        district_summary["district"].astype(str).str.lower() == "unassigned",
                        "n_origins",
                    ].max()
                    if (district_summary["district"].astype(str).str.lower() == "unassigned").any()
                    else 0
                ),
            },
            {
                "metric": "snapping_diagnostic_gt_100m_count",
                "value": int(
                    snapping_diagnostics.loc[
                        snapping_diagnostics["layer"].eq("origins_points_500m"),
                        "flagged_gt_100m_count",
                    ].iloc[0]
                    if not snapping_diagnostics.empty
                    and snapping_diagnostics["layer"].eq("origins_points_500m").any()
                    else 0
                ),
            },
        ]
    )

    all_diagnostics = assignments.drop(columns="geometry").copy()
    unassigned_summary = unassigned.drop(columns="geometry").sort_values("origin_id")

    safe_write_csv(all_diagnostics, TABLES_DIR / "origin_quality_diagnostics_all.csv")
    safe_write_csv(unassigned_summary, TABLES_DIR / "unassigned_origin_diagnostics.csv")
    safe_write_csv(unassigned_detail, TABLES_DIR / "unassigned_origin_overlap_details.csv")
    safe_write_csv(diagnostic_counts, TABLES_DIR / "origin_quality_diagnostic_counts.csv")

    print(
        "Saved origin-quality diagnostics: "
        f"{len(unassigned)} reconstructed previously unassigned origins."
    )


if __name__ == "__main__":
    main()


Overwriting ../src/11_diagnose_origin_quality.py


## `12_assign_districts_by_overlap.py`

Assigns each grid cell to the district with the largest polygon overlap.

In [16]:
%%writefile ../src/12_assign_districts_by_overlap.py
from __future__ import annotations

from config import (
    CRS_METRIC,
    ORIGINS_GRID_FILE,
    ORIGINS_POINTS_FILE,
    PROCESSED_DIR,
    TABLES_DIR,
    ensure_directories,
)
from origin_quality import (
    MIN_DISTRICT_OVERLAP_SHARE,
    attach_assignment_to_points,
    calculate_district_overlaps,
    load_districts,
)
from utils import import_geopandas, require_file, safe_write_csv, safe_write_gdf


GRID_OUTPUT = PROCESSED_DIR / "origins_grid_500m_with_district_overlap.gpkg"
POINTS_OUTPUT = PROCESSED_DIR / "origins_points_500m_with_district_overlap.gpkg"


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()

    grid = gpd.read_file(require_file(ORIGINS_GRID_FILE)).to_crs(CRS_METRIC)
    points = gpd.read_file(require_file(ORIGINS_POINTS_FILE)).to_crs(CRS_METRIC)
    districts = load_districts()

    # Largest-area overlap is more robust than centroid-within assignment for a
    # regular grid clipped to an irregular coastal municipality. Copenhagen
    # contains harbour basins, islands, and boundary-edge cells, so some grid
    # centroids may fall outside a district even though the cell meaningfully
    # overlaps one. Using largest-area overlap avoids losing valid edge cells
    # from district-level summaries.
    assignments, overlap_details = calculate_district_overlaps(
        grid,
        points,
        districts,
        min_overlap_share=MIN_DISTRICT_OVERLAP_SHARE,
    )

    grid_out = grid.merge(assignments.drop(columns="geometry"), on="origin_id", how="left")
    points_out = attach_assignment_to_points(points, assignments)

    safe_write_gdf(grid_out, GRID_OUTPUT, layer="origins_grid_500m_with_district_overlap")
    safe_write_gdf(points_out, POINTS_OUTPUT, layer="origins_points_500m_with_district_overlap")

    required_columns = [
        "origin_id",
        "assigned_district",
        "largest_overlap_area_m2",
        "origin_cell_area_m2",
        "overlap_share",
        "low_overlap_edge_cell",
        "centroid_within_any_district",
        "previous_district_assignment_if_available",
        "district_assignment_changed",
    ]
    diagnostics = assignments.drop(columns="geometry")[required_columns].sort_values("origin_id")
    safe_write_csv(diagnostics, TABLES_DIR / "district_assignment_diagnostics.csv")
    safe_write_csv(overlap_details, TABLES_DIR / "district_assignment_overlap_details.csv")

    print(
        "Saved largest-overlap district assignment for "
        f"{len(assignments)} origin grid cells."
    )


if __name__ == "__main__":
    main()


Overwriting ../src/12_assign_districts_by_overlap.py


## `13_diagnose_snapping_outliers.py`

Finds origins that snap too far from the walking network and maps them.

In [17]:
%%writefile ../src/13_diagnose_snapping_outliers.py
from __future__ import annotations

import pandas as pd

from config import (
    CRS_METRIC,
    FIGURES_DIR,
    MAPS_DIR,
    OSM_WALKING_EDGES_CLEAN_FILE,
    OSM_WALKING_NODES_CLEAN_FILE,
    PROCESSED_DIR,
    TABLES_DIR,
    ensure_directories,
)
from origin_quality import add_snap_flags, calculate_origin_snaps
from utils import import_geopandas, require_file, safe_write_csv, safe_write_gdf


POINTS_WITH_DISTRICT = PROCESSED_DIR / "origins_points_500m_with_district_overlap.gpkg"
SNAP_OUTLIERS_GPKG = PROCESSED_DIR / "origin_snapping_outliers.gpkg"
SNAP_ALL_GPKG = PROCESSED_DIR / "origin_snapping_distances_all.gpkg"


def setup_matplotlib():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def snap_category(distance: float) -> str:
    if distance > 500:
        return ">500 m"
    if distance > 250:
        return "250-500 m"
    if distance > 100:
        return "100-250 m"
    if distance > 50:
        return "50-100 m"
    return "<=50 m"


def save_histogram(snapped) -> None:
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 5))
    snapped["snap_distance_m"].plot(kind="hist", bins=40, color="#4c78a8", edgecolor="white", ax=ax)
    ax.axvline(100, color="#d95f02", linestyle="--", linewidth=1.2, label="100 m")
    ax.axvline(250, color="#7570b3", linestyle="--", linewidth=1.2, label="250 m")
    ax.set_xlabel("Distance to nearest walking-network node (m)")
    ax.set_ylabel("Origin count")
    ax.set_title("Origin snapping distances")
    ax.legend()
    fig.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "origin_snapping_distance_histogram.png", dpi=220, bbox_inches="tight")
    plt.close(fig)


def save_outlier_map(snapped, edges) -> None:
    plt = setup_matplotlib()
    colors = {
        "50-100 m": "#fee08b",
        "100-250 m": "#fdae61",
        "250-500 m": "#d73027",
        ">500 m": "#7f0000",
    }
    flagged = snapped[snapped["snap_gt_50m"]].copy()
    fig, ax = plt.subplots(figsize=(8, 8))
    edges.to_crs(CRS_METRIC).plot(ax=ax, color="#d0d0d0", linewidth=0.25, alpha=0.55)
    for category, color in colors.items():
        subset = flagged[flagged["snap_category"] == category]
        if not subset.empty:
            subset.plot(ax=ax, color=color, markersize=18, label=category, alpha=0.9)
    if not flagged.empty:
        minx, miny, maxx, maxy = flagged.total_bounds
        pad = 800
        ax.set_xlim(minx - pad, maxx + pad)
        ax.set_ylim(miny - pad, maxy + pad)
    ax.set_title("Origin snapping outliers")
    ax.set_axis_off()
    ax.legend(loc="lower left", frameon=True, fontsize=8)
    fig.tight_layout()
    MAPS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(MAPS_DIR / "origin_snapping_outliers_map.png", dpi=220, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "origin_snapping_outliers_map.png", dpi=220, bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()

    points = gpd.read_file(require_file(POINTS_WITH_DISTRICT)).to_crs(CRS_METRIC)
    nodes = gpd.read_file(require_file(OSM_WALKING_NODES_CLEAN_FILE)).to_crs(CRS_METRIC)
    edges = gpd.read_file(require_file(OSM_WALKING_EDGES_CLEAN_FILE)).to_crs(CRS_METRIC)

    # Origin snapping distance is a direct source of accessibility error. If a
    # grid origin is snapped hundreds of metres away from its actual centroid,
    # the calculated 15-minute walking access may represent the network position
    # rather than the intended origin location. The existing diagnostics show 87
    # origins snapped more than 100 m away and a maximum snapping distance of
    # about 882 m, so a cleaning or sensitivity strategy is needed.
    snapped = calculate_origin_snaps(points, nodes)
    snapped = add_snap_flags(snapped)
    snapped["snap_category"] = snapped["snap_distance_m"].apply(snap_category)
    snapped["assigned_district"] = snapped["assigned_district"].fillna("No district overlap")

    flagged = snapped[snapped["snap_gt_50m"]].copy()
    required_columns = [
        "origin_id",
        "assigned_district",
        "snap_distance_m",
        "snap_gt_50m",
        "snap_gt_100m",
        "snap_gt_250m",
        "snap_gt_500m",
        "nearest_node_id",
        "geometry",
    ]

    safe_write_gdf(snapped, SNAP_ALL_GPKG, layer="origin_snapping_distances_all")
    safe_write_gdf(flagged, SNAP_OUTLIERS_GPKG, layer="origin_snapping_outliers")
    safe_write_csv(flagged[required_columns + ["snap_category"]], TABLES_DIR / "origin_snapping_outliers.csv")

    summary = (
        snapped.groupby("assigned_district", dropna=False)
        .agg(
            n_origins=("origin_id", "count"),
            median_snap_distance_m=("snap_distance_m", "median"),
            mean_snap_distance_m=("snap_distance_m", "mean"),
            max_snap_distance_m=("snap_distance_m", "max"),
            n_snap_gt_50m=("snap_gt_50m", "sum"),
            n_snap_gt_100m=("snap_gt_100m", "sum"),
            n_snap_gt_250m=("snap_gt_250m", "sum"),
            n_snap_gt_500m=("snap_gt_500m", "sum"),
        )
        .reset_index()
    )
    for col in ["n_snap_gt_50m", "n_snap_gt_100m", "n_snap_gt_250m", "n_snap_gt_500m"]:
        summary[col] = summary[col].astype(int)
    safe_write_csv(summary, TABLES_DIR / "origin_snapping_summary_by_district.csv")

    save_histogram(snapped)
    save_outlier_map(snapped, edges)

    print(
        "Saved snapping diagnostics: "
        f"{len(flagged)} origins >50 m, {int(snapped['snap_gt_100m'].sum())} origins >100 m."
    )


if __name__ == "__main__":
    main()


Overwriting ../src/13_diagnose_snapping_outliers.py


## `14_create_clean_origin_set.py`

Builds the full, clean100, and clean250 origin sets.

In [18]:
%%writefile ../src/14_create_clean_origin_set.py
from __future__ import annotations

import pandas as pd

from config import (
    CRS_METRIC,
    ORIGINS_GRID_FILE,
    PROCESSED_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import import_geopandas, require_file, safe_write_csv, safe_write_gdf


POINTS_WITH_SNAPS = PROCESSED_DIR / "origin_snapping_distances_all.gpkg"
GRID_WITH_DISTRICT = PROCESSED_DIR / "origins_grid_500m_with_district_overlap.gpkg"


ORIGIN_SET_OUTPUTS = {
    "full": {
        "points": PROCESSED_DIR / "origins_points_500m_full.gpkg",
        "grid": PROCESSED_DIR / "origins_grid_500m_full.gpkg",
    },
    "clean100": {
        "points": PROCESSED_DIR / "origins_points_500m_clean100.gpkg",
        "grid": PROCESSED_DIR / "origins_grid_500m_clean100.gpkg",
    },
    "clean250": {
        "points": PROCESSED_DIR / "origins_points_500m_clean250.gpkg",
        "grid": PROCESSED_DIR / "origins_grid_500m_clean250.gpkg",
    },
}


def valid_district(series: pd.Series) -> pd.Series:
    text = series.fillna("").astype(str).str.strip()
    return (text != "") & (text.str.lower() != "no district overlap") & (text.str.lower() != "unassigned")


def summarize_set(origin_set: str, keep_mask, all_points) -> dict:
    removed = ~keep_mask
    kept = all_points[keep_mask]
    unassigned = ~valid_district(kept["assigned_district"])
    return {
        "origin_set": origin_set,
        "n_origins": int(len(kept)),
        "removed_due_to_snap_gt_100m": int((removed & all_points["snap_gt_100m"]).sum()),
        "removed_due_to_snap_gt_250m": int((removed & all_points["snap_gt_250m"]).sum()),
        "removed_due_to_low_overlap": int((removed & all_points["low_overlap_edge_cell"]).sum()),
        "removed_total": int(removed.sum()),
        "share_removed": float(removed.mean()) if len(all_points) else 0.0,
        "median_snap_distance_m": float(kept["snap_distance_m"].median()) if len(kept) else None,
        "max_snap_distance_m": float(kept["snap_distance_m"].max()) if len(kept) else None,
        "n_unassigned_districts": int(unassigned.sum()),
    }


def save_origin_set(origin_set: str, keep_mask, points, grid, summaries: list[dict]) -> None:
    outputs = ORIGIN_SET_OUTPUTS[origin_set]
    kept_points = points[keep_mask].copy()
    kept_grid = grid[grid["origin_id"].isin(kept_points["origin_id"])].copy()
    safe_write_gdf(kept_points, outputs["points"], layer=outputs["points"].stem)
    safe_write_gdf(kept_grid, outputs["grid"], layer=outputs["grid"].stem)
    summaries.append(summarize_set(origin_set, keep_mask, points))


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()

    points = gpd.read_file(require_file(POINTS_WITH_SNAPS)).to_crs(CRS_METRIC)
    grid_base = gpd.read_file(require_file(GRID_WITH_DISTRICT)).to_crs(CRS_METRIC)
    require_file(ORIGINS_GRID_FILE)

    points["snap_gt_100m"] = points["snap_distance_m"] > 100
    points["snap_gt_250m"] = points["snap_distance_m"] > 250
    points["low_overlap_edge_cell"] = points["overlap_share"] < 0.10
    valid_assignment = valid_district(points["assigned_district"])

    grid_attrs = points.drop(columns="geometry")
    grid = grid_base[["origin_id", "geometry"]].merge(grid_attrs, on="origin_id", how="left")
    grid = gpd.GeoDataFrame(grid, geometry="geometry", crs=CRS_METRIC)

    # Do not simply delete problematic origins without preserving the original
    # baseline. The project should report both the original result and cleaned
    # sensitivity results. This makes the methodology transparent and shows
    # whether conclusions are robust to removing origins that are poorly
    # connected to the walking network or poorly assigned to districts.
    full_keep = pd.Series(True, index=points.index)
    clean100_keep = (~points["snap_gt_100m"]) & (~points["low_overlap_edge_cell"]) & valid_assignment
    clean250_keep = (~points["snap_gt_250m"]) & (~points["low_overlap_edge_cell"]) & valid_assignment

    summaries: list[dict] = []
    save_origin_set("full", full_keep, points, grid, summaries)
    save_origin_set("clean100", clean100_keep, points, grid, summaries)
    save_origin_set("clean250", clean250_keep, points, grid, summaries)

    safe_write_csv(pd.DataFrame(summaries), TABLES_DIR / "origin_cleaning_summary.csv")
    print("Saved full, clean100, and clean250 origin sets.")


if __name__ == "__main__":
    main()


Overwriting ../src/14_create_clean_origin_set.py


## `15_rerun_accessibility_for_clean_origins.py`

Reruns accessibility while changing only the origin set.

In [19]:
%%writefile ../src/15_rerun_accessibility_for_clean_origins.py
from __future__ import annotations

import math

import networkx as nx
import pandas as pd

from config import (
    ACCESS_THRESHOLD_MINUTES,
    AMENITY_LAYER_KEYS,
    AMENITY_TYPE_BY_LAYER,
    CRS_METRIC,
    OSM_WALKING_EDGES_CLEAN_FILE,
    OSM_WALKING_GRAPH_FILE,
    OSM_WALKING_NODES_CLEAN_FILE,
    PROCESSED_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import classify_accessibility, clean_numeric_difference, import_geopandas, require_file, safe_write_csv


ORIGIN_SET_POINTS = {
    "full": PROCESSED_DIR / "origins_points_500m_full.gpkg",
    "clean100": PROCESSED_DIR / "origins_points_500m_clean100.gpkg",
    "clean250": PROCESSED_DIR / "origins_points_500m_clean250.gpkg",
}

ACCESSIBILITY_OUTPUTS = {
    "full": TABLES_DIR / "origin_accessibility_classified_full.csv",
    "clean100": TABLES_DIR / "origin_accessibility_classified_clean100.csv",
    "clean250": TABLES_DIR / "origin_accessibility_classified_clean250.csv",
}

COMPOSITE_OUTPUTS = {
    "full": TABLES_DIR / "composite_disagreement_scores_full.csv",
    "clean100": TABLES_DIR / "composite_disagreement_scores_clean100.csv",
    "clean250": TABLES_DIR / "composite_disagreement_scores_clean250.csv",
}


def load_graph() -> nx.Graph:
    if OSM_WALKING_GRAPH_FILE.exists():
        graph = nx.read_graphml(OSM_WALKING_GRAPH_FILE)
    else:
        gpd = import_geopandas()
        nodes = gpd.read_file(require_file(OSM_WALKING_NODES_CLEAN_FILE)).to_crs(CRS_METRIC)
        edges = gpd.read_file(require_file(OSM_WALKING_EDGES_CLEAN_FILE)).to_crs(CRS_METRIC)
        graph = nx.Graph()
        for row in nodes.itertuples(index=False):
            graph.add_node(str(row.node_id), x=float(row.geometry.x), y=float(row.geometry.y))
        for row in edges.itertuples(index=False):
            graph.add_edge(
                str(row.u),
                str(row.v),
                length_m=float(row.length_m),
                walking_time_minutes=float(row.walking_time_minutes),
            )

    for _, _, data in graph.edges(data=True):
        data["walking_time_minutes"] = float(data["walking_time_minutes"])
    return graph


def node_column(gdf) -> str:
    for col in ["nearest_node_id", "nearest_node", "origin_nearest_node"]:
        if col in gdf.columns:
            return col
    raise ValueError(f"No snapped node column found. Columns: {list(gdf.columns)}")


def load_amenity_nodes():
    gpd = import_geopandas()
    amenity_nodes = {}
    for layer_key in AMENITY_LAYER_KEYS:
        amenity_type = AMENITY_TYPE_BY_LAYER[layer_key]
        for source in ["official", "osm"]:
            path = PROCESSED_DIR / f"{source}_{layer_key}_snapped.gpkg"
            gdf = gpd.read_file(require_file(path)).to_crs(CRS_METRIC)
            col = node_column(gdf)
            amenity_nodes[(source, amenity_type)] = set(gdf[col].dropna().astype(str))
    return amenity_nodes


def shortest_times_to_sources(graph: nx.Graph, source_nodes: set[str]) -> dict[str, float]:
    valid_sources = {node for node in source_nodes if node in graph}
    if not valid_sources:
        return {}
    return nx.multi_source_dijkstra_path_length(
        graph,
        sources=valid_sources,
        weight="walking_time_minutes",
    )


def finite_access(value: float) -> bool:
    return math.isfinite(value) and value <= ACCESS_THRESHOLD_MINUTES


def compute_for_origin_set(origin_set: str, graph, time_cache) -> pd.DataFrame:
    gpd = import_geopandas()
    origins = gpd.read_file(require_file(ORIGIN_SET_POINTS[origin_set])).to_crs(CRS_METRIC)
    col = node_column(origins)
    origins["origin_nearest_node"] = origins[col].astype(str)

    rows = []
    for amenity_type in AMENITY_TYPE_BY_LAYER.values():
        official_lengths = time_cache[("official", amenity_type)]
        osm_lengths = time_cache[("osm", amenity_type)]
        for origin in origins[["origin_id", "origin_nearest_node"]].itertuples(index=False):
            node = str(origin.origin_nearest_node)
            official_time = official_lengths.get(node, float("inf"))
            osm_time = osm_lengths.get(node, float("inf"))
            official_access = finite_access(official_time)
            osm_access = finite_access(osm_time)
            distortion_class = classify_accessibility(official_access, osm_access)
            rows.append(
                {
                    "origin_set": origin_set,
                    "origin_id": origin.origin_id,
                    "amenity_type": amenity_type,
                    "official_nearest_time": official_time if math.isfinite(official_time) else None,
                    "osm_nearest_time": osm_time if math.isfinite(osm_time) else None,
                    "official_access_15": official_access,
                    "osm_access_15": osm_access,
                    "difference_minutes": clean_numeric_difference(osm_time, official_time),
                    "origin_nearest_node": node,
                    "distortion_class": distortion_class,
                    "disagrees": distortion_class in {"osm_false_access", "osm_hidden_access"},
                }
            )
    return pd.DataFrame(rows)


def save_composite(origin_set: str, classified: pd.DataFrame) -> None:
    composite = (
        classified.groupby("origin_id", as_index=False)["disagrees"]
        .sum()
        .rename(columns={"disagrees": "composite_disagreement_score"})
    )
    composite.insert(0, "origin_set", origin_set)
    safe_write_csv(composite, COMPOSITE_OUTPUTS[origin_set])


def main() -> None:
    ensure_directories()
    graph = load_graph()
    amenity_nodes = load_amenity_nodes()

    # The goal is to test whether the main conclusion depends on problematic
    # grid origins. The walking network and POI datasets stay fixed; only the
    # origin set changes. Stable results after removing poorly snapped origins
    # support robustness, while large changes must be discussed as a limitation.
    time_cache = {
        key: shortest_times_to_sources(graph, nodes)
        for key, nodes in amenity_nodes.items()
    }

    for origin_set in ["full", "clean100", "clean250"]:
        classified = compute_for_origin_set(origin_set, graph, time_cache)
        safe_write_csv(classified, ACCESSIBILITY_OUTPUTS[origin_set])
        save_composite(origin_set, classified)
        print(f"Saved accessibility for {origin_set}: {classified['origin_id'].nunique()} origins.")


if __name__ == "__main__":
    main()


Overwriting ../src/15_rerun_accessibility_for_clean_origins.py


## `16_rerun_district_summaries.py`

Rebuilds district summaries using the corrected district assignment.

In [20]:
%%writefile ../src/16_rerun_district_summaries.py
from __future__ import annotations

import pandas as pd

from config import CRS_METRIC, PROCESSED_DIR, TABLES_DIR, ensure_directories
from utils import import_geopandas, require_file, safe_write_csv


ORIGIN_SET_POINTS = {
    "full": PROCESSED_DIR / "origins_points_500m_full.gpkg",
    "clean100": PROCESSED_DIR / "origins_points_500m_clean100.gpkg",
    "clean250": PROCESSED_DIR / "origins_points_500m_clean250.gpkg",
}

ACCESSIBILITY_INPUTS = {
    "full": TABLES_DIR / "origin_accessibility_classified_full.csv",
    "clean100": TABLES_DIR / "origin_accessibility_classified_clean100.csv",
    "clean250": TABLES_DIR / "origin_accessibility_classified_clean250.csv",
}

SUMMARY_OUTPUTS = {
    "full": TABLES_DIR / "district_accessibility_summary_full_corrected.csv",
    "clean100": TABLES_DIR / "district_accessibility_summary_clean100.csv",
    "clean250": TABLES_DIR / "district_accessibility_summary_clean250.csv",
}


def to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def valid_district(series: pd.Series) -> pd.Series:
    text = series.fillna("").astype(str).str.strip()
    return (text != "") & (text.str.lower() != "unassigned") & (text.str.lower() != "no district overlap")


def summarize(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (district, amenity_type), group in data.groupby(["assigned_district", "amenity_type"], dropna=False):
        official_share = group["official_access_15"].mean()
        osm_share = group["osm_access_15"].mean()
        rows.append(
            {
                "district": district,
                "amenity_type": amenity_type,
                "n_origins": int(group["origin_id"].nunique()),
                "official_share_accessible_15": official_share,
                "osm_share_accessible_15": osm_share,
                "difference_share_percentage_points": (osm_share - official_share) * 100.0,
                "share_osm_false_access": (group["distortion_class"] == "osm_false_access").mean(),
                "share_osm_hidden_access": (group["distortion_class"] == "osm_hidden_access").mean(),
                "median_time_difference": group["difference_minutes"].median(),
            }
        )
    return pd.DataFrame(rows).sort_values(["district", "amenity_type"]).reset_index(drop=True)


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()
    remaining_unassigned = []

    # District-level summaries are used for interpretation of spatial inequality
    # across Copenhagen. If many origins are unassigned, district-level results
    # are incomplete and may overrepresent or underrepresent specific areas.
    # Correcting district assignment by largest overlap makes the summaries more
    # reliable.
    for origin_set in ["full", "clean100", "clean250"]:
        origins = gpd.read_file(require_file(ORIGIN_SET_POINTS[origin_set])).to_crs(CRS_METRIC)
        assignments = origins[
            [
                "origin_id",
                "assigned_district",
                "overlap_share",
                "largest_overlap_area_m2",
                "low_overlap_edge_cell",
            ]
        ].copy()
        accessibility = pd.read_csv(require_file(ACCESSIBILITY_INPUTS[origin_set]))
        accessibility["official_access_15"] = accessibility["official_access_15"].apply(to_bool)
        accessibility["osm_access_15"] = accessibility["osm_access_15"].apply(to_bool)
        accessibility["difference_minutes"] = pd.to_numeric(accessibility["difference_minutes"], errors="coerce")

        data = accessibility.merge(assignments, on="origin_id", how="left")
        assigned_mask = valid_district(data["assigned_district"])
        if (~assigned_mask).any():
            remaining = data.loc[
                ~assigned_mask,
                [
                    "origin_set",
                    "origin_id",
                    "amenity_type",
                    "assigned_district",
                    "overlap_share",
                    "largest_overlap_area_m2",
                    "low_overlap_edge_cell",
                ],
            ].drop_duplicates()
            remaining_unassigned.append(remaining)

        summary = summarize(data[assigned_mask].copy())
        safe_write_csv(summary, SUMMARY_OUTPUTS[origin_set])
        print(
            f"Saved corrected district summary for {origin_set}: "
            f"{summary['district'].nunique() if not summary.empty else 0} districts."
        )

    if remaining_unassigned:
        out = pd.concat(remaining_unassigned, ignore_index=True)
    else:
        out = pd.DataFrame(
            columns=[
                "origin_set",
                "origin_id",
                "amenity_type",
                "assigned_district",
                "overlap_share",
                "largest_overlap_area_m2",
                "low_overlap_edge_cell",
            ]
        )
    safe_write_csv(out, TABLES_DIR / "remaining_unassigned_origins.csv")


if __name__ == "__main__":
    main()


Overwriting ../src/16_rerun_district_summaries.py


## `17_compare_baseline_vs_cleaned.py`

Compares the baseline and cleaned results and writes a short robustness interpretation.

In [21]:
%%writefile ../src/17_compare_baseline_vs_cleaned.py
from __future__ import annotations

import pandas as pd

from config import FIGURES_DIR, TABLES_DIR, ensure_directories
from utils import require_file, safe_write_csv


ACCESSIBILITY_INPUTS = {
    "full": TABLES_DIR / "origin_accessibility_classified_full.csv",
    "clean100": TABLES_DIR / "origin_accessibility_classified_clean100.csv",
    "clean250": TABLES_DIR / "origin_accessibility_classified_clean250.csv",
}

DISTRICT_INPUTS = {
    "full": TABLES_DIR / "district_accessibility_summary_full_corrected.csv",
    "clean100": TABLES_DIR / "district_accessibility_summary_clean100.csv",
    "clean250": TABLES_DIR / "district_accessibility_summary_clean250.csv",
}


def to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def load_accessibility(origin_set: str) -> pd.DataFrame:
    df = pd.read_csv(require_file(ACCESSIBILITY_INPUTS[origin_set]))
    df["origin_set"] = origin_set
    df["official_access_15"] = df["official_access_15"].apply(to_bool)
    df["osm_access_15"] = df["osm_access_15"].apply(to_bool)
    df["disagrees"] = df["disagrees"].apply(to_bool)
    df["difference_minutes"] = pd.to_numeric(df["difference_minutes"], errors="coerce")
    return df


def robustness_summary() -> pd.DataFrame:
    rows = []
    for origin_set in ["full", "clean100", "clean250"]:
        df = load_accessibility(origin_set)
        for amenity_type, group in df.groupby("amenity_type"):
            official_share = group["official_access_15"].mean()
            osm_share = group["osm_access_15"].mean()
            rows.append(
                {
                    "origin_set": origin_set,
                    "amenity_type": amenity_type,
                    "n_origins": int(group["origin_id"].nunique()),
                    "official_share_accessible_15": official_share,
                    "osm_share_accessible_15": osm_share,
                    "difference_share_percentage_points": (osm_share - official_share) * 100.0,
                    "total_disagreement_share": group["disagrees"].mean(),
                    "osm_false_access_share": (group["distortion_class"] == "osm_false_access").mean(),
                    "osm_hidden_access_share": (group["distortion_class"] == "osm_hidden_access").mean(),
                    "median_difference_minutes": group["difference_minutes"].median(),
                    "mean_difference_minutes": group["difference_minutes"].mean(),
                }
            )
    return pd.DataFrame(rows)


def district_change_table() -> pd.DataFrame:
    full = pd.read_csv(require_file(DISTRICT_INPUTS["full"]))[
        ["district", "amenity_type", "difference_share_percentage_points"]
    ].rename(columns={"difference_share_percentage_points": "original_difference_share_percentage_points"})
    clean100 = pd.read_csv(require_file(DISTRICT_INPUTS["clean100"]))[
        ["district", "amenity_type", "difference_share_percentage_points"]
    ].rename(columns={"difference_share_percentage_points": "clean100_difference_share_percentage_points"})
    clean250 = pd.read_csv(require_file(DISTRICT_INPUTS["clean250"]))[
        ["district", "amenity_type", "difference_share_percentage_points"]
    ].rename(columns={"difference_share_percentage_points": "clean250_difference_share_percentage_points"})

    out = full.merge(clean100, on=["district", "amenity_type"], how="outer").merge(
        clean250,
        on=["district", "amenity_type"],
        how="outer",
    )
    out["change_after_cleaning"] = (
        out["clean100_difference_share_percentage_points"]
        - out["original_difference_share_percentage_points"]
    )
    out["clean250_change_after_cleaning"] = (
        out["clean250_difference_share_percentage_points"]
        - out["original_difference_share_percentage_points"]
    )
    return out.sort_values(["district", "amenity_type"]).reset_index(drop=True)


def save_barplot(summary: pd.DataFrame) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    order = ["library", "playground", "sports_facility"]
    sets = ["full", "clean100", "clean250"]
    pivot = summary.pivot(index="amenity_type", columns="origin_set", values="difference_share_percentage_points")
    pivot = pivot.reindex(order)

    x = range(len(pivot.index))
    width = 0.24
    fig, ax = plt.subplots(figsize=(8, 5))
    offsets = {"full": -width, "clean100": 0.0, "clean250": width}
    colors = {"full": "#4c78a8", "clean100": "#f58518", "clean250": "#54a24b"}
    for origin_set in sets:
        values = pivot[origin_set].to_numpy()
        ax.bar([idx + offsets[origin_set] for idx in x], values, width=width, label=origin_set, color=colors[origin_set])
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel("OSM minus official accessible share (percentage points)")
    ax.set_title("Robustness of OSM-official accessibility difference")
    ax.legend()
    fig.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "robustness_accessibility_difference_barplot.png", dpi=220, bbox_inches="tight")
    plt.close(fig)


def same_direction(a: float, b: float) -> bool:
    if pd.isna(a) or pd.isna(b):
        return False
    if abs(a) < 1 and abs(b) < 1:
        return True
    return (a >= 0 and b >= 0) or (a <= 0 and b <= 0)


def write_interpretation(summary: pd.DataFrame, district_changes: pd.DataFrame) -> None:
    full = summary[summary["origin_set"] == "full"].set_index("amenity_type")
    clean = summary[summary["origin_set"] == "clean100"].set_index("amenity_type")
    rows = []
    affected = []
    for amenity_type in full.index:
        original = full.loc[amenity_type, "difference_share_percentage_points"]
        cleaned = clean.loc[amenity_type, "difference_share_percentage_points"]
        change = cleaned - original
        stable = same_direction(original, cleaned) and abs(change) <= 5.0
        rows.append((amenity_type, original, cleaned, change, stable))
        if not stable:
            affected.append(amenity_type)

    lines = ["# Robustness interpretation", ""]
    if not affected:
        lines.append(
            "The direction and approximate size of the main findings remain stable after "
            "clean100 origin-quality filtering, so the results are robust to origin-quality filtering."
        )
    else:
        lines.append(
            "The cleaned results change substantially for "
            + ", ".join(affected)
            + ". This indicates that some original disagreement was driven by edge cells, "
            "harbour/water-adjacent cells, or poorly snapped origins."
        )
    lines.append("")
    lines.append("A change larger than 5 percentage points or a direction change is treated as substantial.")
    lines.append("")
    lines.append("| Amenity type | Full pp difference | Clean100 pp difference | Change pp | Stable |")
    lines.append("|---|---:|---:|---:|---|")
    for amenity_type, original, cleaned, change, stable in rows:
        lines.append(f"| {amenity_type} | {original:.2f} | {cleaned:.2f} | {change:.2f} | {stable} |")

    district_changes = district_changes.copy()
    district_changes["abs_change_after_cleaning"] = district_changes["change_after_cleaning"].abs()
    top = district_changes.sort_values("abs_change_after_cleaning", ascending=False).head(10)
    lines.extend(["", "## Districts most affected by clean100 filtering", ""])
    lines.append("| District | Amenity type | Change pp |")
    lines.append("|---|---|---:|")
    for row in top.itertuples(index=False):
        lines.append(f"| {row.district} | {row.amenity_type} | {row.change_after_cleaning:.2f} |")

    output = TABLES_DIR / "robustness_interpretation.md"
    output.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> None:
    ensure_directories()
    summary = robustness_summary()
    district_changes = district_change_table()
    safe_write_csv(summary, TABLES_DIR / "robustness_summary_by_origin_set.csv")
    safe_write_csv(district_changes, TABLES_DIR / "district_robustness_comparison.csv")
    save_barplot(summary)
    write_interpretation(summary, district_changes)
    print("Saved robustness summaries, comparison table, barplot, and interpretation notes.")


if __name__ == "__main__":
    main()


Overwriting ../src/17_compare_baseline_vs_cleaned.py


## `18_make_cleaned_maps.py`

Makes the recommended clean100 maps.

In [22]:
%%writefile ../src/18_make_cleaned_maps.py
from __future__ import annotations

import pandas as pd

from config import (
    BYDELE_FILE,
    CRS_METRIC,
    FIGURES_DIR,
    OSM_WALKING_EDGES_CLEAN_FILE,
    PROCESSED_DIR,
    TABLES_DIR,
    ensure_directories,
)
from utils import import_geopandas, require_file


CLASSIFIED_CLEAN100 = TABLES_DIR / "origin_accessibility_classified_clean100.csv"
COMPOSITE_CLEAN100 = TABLES_DIR / "composite_disagreement_scores_clean100.csv"
GRID_CLEAN100 = PROCESSED_DIR / "origins_grid_500m_clean100.gpkg"
SNAP_OUTLIERS = PROCESSED_DIR / "origin_snapping_outliers.gpkg"


ACCESS_COLORS = {True: "#2ca25f", False: "#f0f0f0"}
DISAGREEMENT_COLORS = {
    "agreement_accessible": "#2ca25f",
    "agreement_inaccessible": "#d9d9d9",
    "osm_false_access": "#d73027",
    "osm_hidden_access": "#4575b4",
}


def setup_matplotlib():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def add_basemap(ax, crs) -> None:
    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs=crs, source=ctx.providers.CartoDB.PositronNoLabels, attribution_size=6)
    except Exception:
        return


def finish(fig, ax, output, title):
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()
    fig.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=220, bbox_inches="tight")
    import matplotlib.pyplot as plt

    plt.close(fig)


def to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def plot_access(grid, boundary, amenity_type, source, column):
    plt = setup_matplotlib()
    from matplotlib.patches import Patch

    fig, ax = plt.subplots(figsize=(8, 8))
    for value, color in ACCESS_COLORS.items():
        subset = grid[grid[column] == value]
        if not subset.empty:
            subset.plot(
                ax=ax,
                color=color,
                edgecolor="white",
                linewidth=0.15,
                label="accessible" if value else "not accessible",
            )
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    handles = [
        Patch(facecolor=ACCESS_COLORS[True], edgecolor="white", label="accessible"),
        Patch(facecolor=ACCESS_COLORS[False], edgecolor="white", label="not accessible"),
    ]
    ax.legend(handles=handles, loc="lower left", frameon=True)
    add_basemap(ax, grid.crs)
    output = FIGURES_DIR / f"clean100_{source}_{amenity_type}_access_15min.png"
    finish(fig, ax, output, f"Clean100 {source} 15-minute access: {amenity_type}")


def plot_disagreement(grid, boundary, amenity_type):
    plt = setup_matplotlib()
    from matplotlib.patches import Patch

    fig, ax = plt.subplots(figsize=(8, 8))
    for status, color in DISAGREEMENT_COLORS.items():
        subset = grid[grid["distortion_class"] == status]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.15, label=status)
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    handles = [
        Patch(facecolor=color, edgecolor="white", label=status)
        for status, color in DISAGREEMENT_COLORS.items()
        if not grid[grid["distortion_class"] == status].empty
    ]
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)
    add_basemap(ax, grid.crs)
    finish(fig, ax, FIGURES_DIR / f"clean100_{amenity_type}_disagreement.png", f"Clean100 disagreement: {amenity_type}")


def plot_difference(grid, boundary, amenity_type):
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    grid.plot(
        ax=ax,
        column="difference_minutes",
        cmap="RdBu_r",
        legend=True,
        edgecolor="white",
        linewidth=0.15,
        missing_kwds={"color": "#efefef", "label": "no route"},
    )
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    add_basemap(ax, grid.crs)
    finish(
        fig,
        ax,
        FIGURES_DIR / f"clean100_{amenity_type}_walking_time_difference.png",
        f"Clean100 walking-time difference: {amenity_type}",
    )


def plot_composite(grid, boundary, composite):
    plt = setup_matplotlib()
    fig, ax = plt.subplots(figsize=(8, 8))
    mapped = grid.merge(composite, on="origin_id", how="left")
    mapped["composite_disagreement_score"] = mapped["composite_disagreement_score"].fillna(0).astype(int)
    mapped.plot(
        ax=ax,
        column="composite_disagreement_score",
        cmap="YlOrRd",
        vmin=0,
        vmax=3,
        legend=True,
        edgecolor="white",
        linewidth=0.15,
    )
    boundary.boundary.plot(ax=ax, color="#222222", linewidth=0.7)
    add_basemap(ax, mapped.crs)
    finish(fig, ax, FIGURES_DIR / "clean100_composite_disagreement_score.png", "Clean100 composite disagreement score")


def plot_snapping_outliers():
    gpd = import_geopandas()
    if not SNAP_OUTLIERS.exists():
        print("Snapping outlier GeoPackage not found; skipping cleaned snapping outlier map.")
        return
    outliers = gpd.read_file(SNAP_OUTLIERS).to_crs(CRS_METRIC)
    edges = gpd.read_file(require_file(OSM_WALKING_EDGES_CLEAN_FILE)).to_crs(CRS_METRIC)
    if outliers.empty:
        return

    plt = setup_matplotlib()
    colors = {
        "50-100 m": "#fee08b",
        "100-250 m": "#fdae61",
        "250-500 m": "#d73027",
        ">500 m": "#7f0000",
    }
    fig, ax = plt.subplots(figsize=(8, 8))
    edges.plot(ax=ax, color="#d0d0d0", linewidth=0.25, alpha=0.55)
    for category, color in colors.items():
        subset = outliers[outliers["snap_category"] == category]
        if not subset.empty:
            subset.plot(ax=ax, color=color, markersize=18, label=category, alpha=0.9)
    minx, miny, maxx, maxy = outliers.total_bounds
    pad = 800
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)
    ax.legend(loc="lower left", frameon=True, fontsize=8)
    finish(fig, ax, FIGURES_DIR / "origin_snapping_outliers_map.png", "Origin snapping outliers")


def main() -> None:
    ensure_directories()
    gpd = import_geopandas()

    classified = pd.read_csv(require_file(CLASSIFIED_CLEAN100))
    for column in ["official_access_15", "osm_access_15"]:
        classified[column] = classified[column].apply(to_bool)

    grid = gpd.read_file(require_file(GRID_CLEAN100)).to_crs(CRS_METRIC)
    boundary = gpd.read_file(require_file(BYDELE_FILE)).to_crs(CRS_METRIC)
    composite = pd.read_csv(require_file(COMPOSITE_CLEAN100))

    cleaning_summary = pd.read_csv(TABLES_DIR / "origin_cleaning_summary.csv") if (TABLES_DIR / "origin_cleaning_summary.csv").exists() else pd.DataFrame()
    if not cleaning_summary.empty:
        clean100_share = cleaning_summary.loc[cleaning_summary["origin_set"] == "clean100", "share_removed"]
        if not clean100_share.empty and float(clean100_share.iloc[0]) > 0.30:
            print("Warning: clean100 removes more than 30% of origins; inspect robustness outputs before final reporting.")

    # The final report should use maps based on the cleaned and corrected origin
    # set, because these better represent valid land-based origins and corrected
    # district assignment. The baseline results are preserved and checked through
    # robustness analysis.
    for amenity_type, group in classified.groupby("amenity_type"):
        mapped = grid.merge(group, on="origin_id", how="left")
        mapped = gpd.GeoDataFrame(mapped, geometry="geometry", crs=CRS_METRIC)
        plot_access(mapped, boundary, amenity_type, "official", "official_access_15")
        plot_access(mapped, boundary, amenity_type, "osm", "osm_access_15")
        plot_disagreement(mapped, boundary, amenity_type)
        plot_difference(mapped, boundary, amenity_type)

    plot_composite(grid, boundary, composite)
    plot_snapping_outliers()
    print("Saved clean100 maps.")


if __name__ == "__main__":
    main()


Overwriting ../src/18_make_cleaned_maps.py


## `run_pipeline.py`

Runs the numbered scripts in the right order from the command line.

In [23]:
%%writefile ../src/run_pipeline.py
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


SCRIPT_DIR = Path(__file__).resolve().parent

PIPELINE = [
    "01_download_official_data.py",
    "03_clean_official_amenities.py",
    "02_extract_osm_data.py",
    "04_clean_osm_amenities.py",
    "05_prepare_walking_network.py",
    "06_create_origin_grid.py",
    "07_match_osm_to_official.py",
    "08_compute_accessibility.py",
    "09_compare_accessibility.py",
    "10_make_maps.py",
    "11_diagnose_origin_quality.py",
    "12_assign_districts_by_overlap.py",
    "13_diagnose_snapping_outliers.py",
    "14_create_clean_origin_set.py",
    "15_rerun_accessibility_for_clean_origins.py",
    "16_rerun_district_summaries.py",
    "17_compare_baseline_vs_cleaned.py",
    "18_make_cleaned_maps.py",
]


def main() -> None:
    for script in PIPELINE:
        path = SCRIPT_DIR / script
        print(f"\n=== Running {script} ===")
        subprocess.run([sys.executable, str(path)], check=True)


if __name__ == "__main__":
    main()


Overwriting ../src/run_pipeline.py
